In [82]:
# José Afonso da Silva Domingues de Moura (2021273987)
# João Nuno Souteiro Bugalho Coelho (2021275030)

# Breast Cancer Classification using Blood Biomarkers
# Dataset: 116 patients, 9 biomarkers
# Methods: 7 classifiers with feature selection and dimensionality reduction
# Validation: Monte Carlo CV (150 iterations, 70/30 split)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import scipy.stats as stats
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, auc, roc_auc_score
import plotly.express as px
from sklearn.decomposition import PCA   
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


In [83]:
# Constants
RAND_SEED = 42
DATASET_PATH = '../Datasets/'
DATASET_NAME = 'dataR2.xlsx'
LABELS_KEY = 'Classification'
TRAIN_SIZE = 0.7
TEST_SIZE = 0.3

## Dataset load

Reading Excel and dropping rows with Nan values

In [84]:
def load_data(dataset_path=DATASET_PATH, dataset_name=DATASET_NAME):
    """
    Load breast cancer biomarker dataset from Excel file.
    
    Reads Excel file containing blood biomarker measurements and removes
    any rows with missing values (NaN) to ensure clean data for analysis.
    
    Args:
        dataset_path (str): Directory path containing dataset file
        dataset_name (str): Name of Excel file to load
    
    Returns:
        pd.DataFrame: Clean dataset with no missing values
    """
    xls_file = pd.read_excel(dataset_path + dataset_name)
    blood_data = pd.DataFrame(xls_file)
    return blood_data.dropna()

blood_data = load_data()
blood_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             116 non-null    int64  
 1   BMI             116 non-null    float64
 2   Glucose         116 non-null    int64  
 3   Insulin         116 non-null    float64
 4   HOMA            116 non-null    float64
 5   Leptin          116 non-null    float64
 6   Adiponectin     116 non-null    float64
 7   Resistin        116 non-null    float64
 8   MCP.1           116 non-null    float64
 9   Classification  116 non-null    int64  
dtypes: float64(7), int64(3)
memory usage: 9.2 KB


### Data partitioning
Training, validation and testing data partitioning

In [85]:
def normalize(data, train_data, labels_key=LABELS_KEY):
    """
    Normalize data using training set statistics (z-score normalization).
    
    Applies feature scaling by subtracting mean and dividing by standard deviation.
    Critically, uses only training set statistics to prevent data leakage. This
    ensures test set remains truly unseen during model development.
    
    Formula: x_normalized = (x - μ_train) / σ_train
    
    Args:
        data (pd.DataFrame): Data to normalize (can be train or test set)
        train_data (pd.DataFrame): Training data used to compute statistics
        labels_key (str): Label column name to exclude from normalization
    
    Returns:
        pd.DataFrame: Normalized data with same structure as input
    
    Note:
        - Only feature columns are normalized, not labels
        - Prevents information leakage from test to train set
        - Critical for proper model validation
    """
    # Calculate statistics only on features (excluding labels)
    features = [col for col in train_data.columns if col != labels_key]
    std = train_data[features].std(axis=0)
    mean = train_data[features].mean(axis=0)
    
    normalized = data.copy()
    for fname in features:
        normalized[fname] = (data[fname] - mean[fname]) / std[fname]
    
    return normalized


def prepare_data(blood_data, labels_key=LABELS_KEY, random_state=RAND_SEED, 
                 train_size=TRAIN_SIZE, test_size=TEST_SIZE):
    """
    Split dataset into train/test sets and normalize using train statistics.
    
    Implements proper data partitioning for machine learning validation.
    Splits data randomly, then normalizes both sets using only training
    set statistics to prevent data leakage.
    
    Args:
        blood_data (pd.DataFrame): Complete dataset
        labels_key (str): Column name for labels
        random_state (int): Random seed for reproducibility
        train_size (float): Training set proportion (0-1)
        test_size (float): Testing set proportion (0-1)
    
    Returns:
        tuple: (train_data, test_data) both normalized DataFrames
    """
    train_data, test_data = train_test_split(
        blood_data, 
        train_size=train_size, 
        test_size=test_size, 
        random_state=random_state, 
        shuffle=True
    )
    
    # Normalize test set using train statistics (prevents leakage)
    test_data = normalize(test_data, train_data, labels_key)
    # Normalize train set using its own statistics
    train_data = normalize(train_data, train_data, labels_key)
    
    return train_data, test_data


def generate_testing_data(blood_data, labels_key=LABELS_KEY, random_state=RAND_SEED, 
                          train_size=TRAIN_SIZE, test_size=TEST_SIZE):
    """
    Generate normalized train/test splits in numpy array format.
    
    Convenience function that combines data splitting, normalization, and
    conversion to numpy arrays ready for sklearn classifiers.
    
    Args:
        blood_data (pd.DataFrame): Complete dataset
        labels_key (str): Label column name
        random_state (int): Random seed for reproducibility
        train_size (float): Training set proportion
        test_size (float): Testing set proportion
    
    Returns:
        tuple: (x_train, y_train, x_test, y_test) as numpy arrays
            - x_train: Training features, shape (n_train, n_features)
            - y_train: Training labels, shape (n_train,)
            - x_test: Test features, shape (n_test, n_features)
            - y_test: Test labels, shape (n_test,)
    """
    train_data, test_data = prepare_data(
        blood_data, labels_key, random_state, train_size, test_size
    )
    
    x_tr = train_data.drop(columns=[labels_key]).to_numpy()
    y_tr = train_data[labels_key].to_numpy()
    x_test = test_data.drop(columns=[labels_key]).to_numpy()
    y_test = test_data[labels_key].to_numpy()
    
    return x_tr, y_tr, x_test, y_test

## Feature Selection

In [86]:
# Create train/test split for feature analysis
train_data, test_data = train_test_split(
    blood_data,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    random_state=RAND_SEED,
    shuffle=True
)

# Normalize using train statistics
test_data = normalize(test_data, train_data, labels_key=LABELS_KEY)
train_data = normalize(train_data, train_data, labels_key=LABELS_KEY)

print(f"Train set: {len(train_data)} rows, {train_data.shape[1]} cols")
print(f"Test set: {len(test_data)} rows, {test_data.shape[1]} cols")

Train set: 81 rows, 10 cols
Test set: 35 rows, 10 cols


#### Kruskal-Wallis Test

In [87]:
# Kruskal-Wallis Test for feature selection
# Non-parametric test to identify features with significant class differences
labels = np.unique(train_data[LABELS_KEY])
fnames = train_data.drop(columns=[LABELS_KEY]).columns.to_numpy()
class_idxs = [np.where(train_data[LABELS_KEY] == i)[0] for i in labels]

kruskal_ranking = {}
for i in range(fnames.shape[0]):
    # Test if feature values differ significantly between classes
    stat = stats.kruskal(
        *[train_data.iloc[idxs, i].to_numpy().flatten() for idxs in class_idxs]
    )
    kruskal_ranking[fnames[i]] = stat.statistic

# Sort features by test statistic (higher = more discriminative)
kruskal_ranking = sorted(kruskal_ranking.items(), key=lambda x: x[1], reverse=True)
print("Kruskal-Wallis Ranking:", kruskal_ranking)

Kruskal-Wallis Ranking: [('Glucose', 21.187315007601605), ('HOMA', 8.961914861384628), ('Insulin', 5.279469860342602), ('Resistin', 3.7457051961823993), ('BMI', 0.8908951998281452), ('Age', 0.42692296270615), ('MCP.1', 0.3494055512277902), ('Adiponectin', 0.2553278338706653), ('Leptin', 0.0008180578700489605)]


#### ROC-AUC

In [88]:
# ROC-AUC Analysis for feature ranking
# Evaluates each feature's individual classification power
y = train_data[LABELS_KEY].to_numpy()
features = [c for c in train_data.columns if c != LABELS_KEY]

roc_list = []
for f in features:
    x = train_data[f].to_numpy()
    
    # Compute ROC curve for this feature alone
    fpr, tpr, _ = roc_curve(y, x, pos_label=2)
    a = float(auc(fpr, tpr))
    
    # Ensure AUC >= 0.5 (flip if needed)
    if a < 0.5:
        a = 1.0 - a
        fpr, tpr = tpr, fpr
    
    roc_list.append((f, fpr, tpr, a))

# Sort features by AUC (descending)
roc_sorted = sorted(roc_list, key=lambda r: r[3], reverse=True)

# Plot top K ROC curves
K = min(9, len(roc_sorted))
fig = go.Figure()

for feat, fpr, tpr, a in roc_sorted[:K]:
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f"{feat} (AUC={a:.3f})",
        line=dict(width=2)
    ))

# Add diagonal reference line (random classifier)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(dash='dash', color='black'),
    name='Random (AUC=0.5)'
))

fig.update_layout(
    title=f'ROC Curves - Top {K} Features',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=700
)
fig.show()

print("\nTop features by AUC:")
for feat, _, _, a in roc_sorted[:10]:
    print(f' - {feat}: {a:.3f}')

Top features by AUC:
 - Glucose: 0.800
 - HOMA: 0.695
 - Insulin: 0.650
 - Resistin: 0.626
 - BMI: 0.561
 - Age: 0.543
 - MCP.1: 0.539
 - Adiponectin: 0.533
 - Leptin: 0.502


#### Correlation-based feature selection

##### Correlation heatmap

In [89]:
# Correlation Matrix Heatmap
# Visualizes pairwise correlations between all features
feature_names = [f for f in train_data.columns]
X = np.array([train_data[f].to_numpy() for f in feature_names])
corr_matrix = np.corrcoef(X)

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    labels=dict(x="Features", y="Features", color="Correlation"),
    x=feature_names,
    y=feature_names,
    width=800,
    height=800,
    color_continuous_scale='Greens'
)
fig.show()

##### Evaluating correlations and eliminating redundant features

In [90]:
def corr_feature_selection(data, corr_thresh=0.9, top=None):
    """
    Select non-redundant features by removing highly correlated feature pairs.
    
    This function implements correlation-based feature selection to reduce dimensionality
    while preserving predictive power. It iteratively selects features based on their
    correlation with the target variable and removes redundant features that are highly
    correlated with already-selected features.
    
    Algorithm:
    1. Compute correlation matrix for all features and target variable
    2. Rank features by absolute correlation with target (descending order)
    3. For each feature in ranking:
       a. If not already marked as redundant
       b. Check correlation with all lower-ranked features
       c. Mark highly correlated features (>threshold) as redundant
       d. Add current feature to selection
    4. Stop when desired number of features is reached
    
    Args:
        data (pd.DataFrame): DataFrame containing features and target variable.
                            Last column is assumed to be the target.
        corr_thresh (float): Correlation threshold for redundancy detection.
                            Features with |correlation| > threshold are considered redundant.
                            Default: 0.9 (removes features with >90% correlation)
        top (int, optional): Maximum number of features to select.
                            If None, selects all non-redundant features.
                            Default: None (select all non-redundant)
    
    Returns:
        np.ndarray: Array of selected feature names in order of importance.
                   Features are ranked by their correlation with the target.
    
    Note:
        - Higher corr_thresh (e.g., 0.95) keeps more features (less aggressive)
        - Lower corr_thresh (e.g., 0.7) removes more redundant features (more aggressive)
        - First feature in each correlated pair is kept (higher target correlation)
        - Useful for reducing multicollinearity in linear models
    
    Example:
        >>> # Select top 5 non-redundant features with correlation threshold 0.85
        >>> selected_features = corr_feature_selection(train_data, corr_thresh=0.85, top=5)
        >>> print(selected_features)  # ['Glucose', 'BMI', 'Age', 'Leptin', 'Resistin']
    """

    if top is None:
        top = data.shape[1] - 1

    corr_df = data.corr()
    feature_corr = corr_df.iloc[:-1, :-1]  # correlations between features
    target_corr = corr_df.iloc[:-1, -1]    # correlations with the target
    features = feature_corr.columns.tolist()
    selected_features = []
    ignore_feature = [False] * len(features)
    sorted_features = sorted(features, key=lambda f: abs(target_corr[f]), reverse=True)
    for i, feat in enumerate(sorted_features):
        if ignore_feature[i]:
            continue
        insert = True
        for j in range(i + 1, len(sorted_features)):
            if ignore_feature[j]:
                continue
            if abs(feature_corr.loc[feat, sorted_features[j]]) > corr_thresh:
                # the first has better correlation with target!
                ignore_feature[j] = True
        if insert:
            selected_features.append(feat)
            if len(selected_features) == top:
                break


    return np.array(selected_features)

corr_features = corr_feature_selection(train_data.drop(columns=[LABELS_KEY]))
print("Original Features: (" + str(len(train_data.drop(columns=[LABELS_KEY]).columns.tolist())) + " features)")
print(train_data.drop(columns=[LABELS_KEY]).columns.tolist())

print("Ranking of features by absolute correlation with target:")
print(np.abs(train_data.corr()[LABELS_KEY]).sort_values(ascending=False))

Original Features: (9 features)
['Age', 'BMI', 'Glucose', 'Insulin', 'HOMA', 'Leptin', 'Adiponectin', 'Resistin', 'MCP.1']
Ranking of features by absolute correlation with target:
Classification    1.000000
Glucose           0.391128
Insulin           0.281391
HOMA              0.280814
Resistin          0.130698
BMI               0.120082
MCP.1             0.083286
Age               0.064279
Adiponectin       0.019910
Leptin            0.014102
Name: Classification, dtype: float64


## Feature Reduction

#### PCA

In [91]:
def fit_pca_reference(X, n_components=None, labels_key=LABELS_KEY):
    """
    Fit Principal Component Analysis (PCA) on training data.
    
    PCA finds orthogonal directions of maximum variance in the data.
    This function should only be fitted on training data to prevent leakage.
    
    Args:
        X (pd.DataFrame or np.ndarray): Training data
        n_components (int): Number of components to retain
        labels_key (str): Label column to exclude
    
    Returns:
        sklearn.decomposition.PCA: Fitted PCA transformer
    """
    if labels_key in X.columns:
        X = X.drop(columns=[labels_key]).astype(float)
    
    pca = PCA(n_components=n_components)
    pca.fit(X)
    return pca


def pca_feature_reduction(data, n_components=3, labels_key=LABELS_KEY, pca=None):
    """
    Apply PCA transformation to reduce data dimensionality.
    
    Projects data onto principal component space. Can either fit new PCA
    or use pre-fitted transformer (correct approach for test data).
    
    Args:
        data (pd.DataFrame): Data to transform
        n_components (int): Number of components
        labels_key (str): Label column to exclude
        pca (PCA, optional): Pre-fitted PCA transformer
    
    Returns:
        tuple: (pca_components, explained_variance_ratio, pca_object)
            - pca_components: Transformed data in PC space
            - explained_variance_ratio: Variance explained by each PC
            - pca_object: The PCA transformer
    """
    if labels_key in data.columns:
        data = data.drop(columns=[labels_key]).astype(float)
    
    # Fit new PCA if not provided
    if pca is None:
        pca = fit_pca_reference(data, n_components=n_components)
    
    # Transform data using PCA
    pca_comp = pca.transform(data)
    explained_var_ratio = pca.explained_variance_ratio_
    
    return pca_comp, explained_var_ratio, pca

In [92]:
# Apply PCA to full dataset
feat_cols = [c for c in train_data.columns if c != LABELS_KEY]
pca_ref = fit_pca_reference(train_data[feat_cols], n_components=9)

X_train_pca = pca_ref.transform(train_data[feat_cols])
X_test_pca = pca_ref.transform(test_data[feat_cols])

print('Explained variance ratio:', pca_ref.explained_variance_ratio_)
print('Cumulative explained variance:', pca_ref.explained_variance_ratio_.cumsum())

Explained variance ratio: [0.33737429 0.18332678 0.12512476 0.11182853 0.08681259 0.07464767
 0.04323565 0.03501875 0.00263098]
Cumulative explained: [0.33737429 0.52070107 0.64582583 0.75765436 0.84446695 0.91911462
 0.96235027 0.99736902 1.        ]


In [93]:
# 3D PCA Visualization
fig = px.scatter_3d(
    x=X_train_pca[:, 0],
    y=X_train_pca[:, 1],
    z=X_train_pca[:, 2],
    color=train_data[LABELS_KEY].astype(str),
    labels={'x': 'PC1', 'y': 'PC2', 'z': 'PC3'}
)
fig.update_traces(marker_size=4)
fig.show()

In [94]:
# 2D PCA visualization
pca_comp, explained_var_ratio, pca = pca_feature_reduction(train_data, n_components=2)

print('Explained variance ratio (2 components):', explained_var_ratio)

# Plot 2D projection
fig = px.scatter(
    x=pca_comp[:, 0],
    y=pca_comp[:, 1],
    color=train_data[LABELS_KEY].astype(str),
    labels={'x': 'PC1', 'y': 'PC2'}
)
fig.update_traces(marker_size=6)
fig.show()

Explained variance ratio (2 components): [0.33737429 0.18332678]


In [95]:
# Analyze explained variance for all 9 components
pca_comp, explained_var_ratio, pca = pca_feature_reduction(train_data, n_components=9)

print("Cumulative Explained Variance Ratio:")
print(explained_var_ratio.cumsum())
print("\nExplained Variance Ratio per Component:")
print(explained_var_ratio)

Cumulative Explained Variance Ratio:
[0.33737429 0.52070107 0.64582583 0.75765436 0.84446695 0.91911462
 0.96235027 0.99736902 1.        ]
Explained Variance Ratio:
[0.33737429 0.18332678 0.12512476 0.11182853 0.08681259 0.07464767
 0.04323565 0.03501875 0.00263098]


#### LDA feature reduction

In [96]:
def fit_lda_reference(X, y, n_components=1):
    """
    Fit Linear Discriminant Analysis (LDA) for supervised dimensionality reduction.
    
    LDA finds linear combinations of features that best separate classes.
    Unlike PCA (unsupervised), LDA uses class labels to maximize between-class
    variance while minimizing within-class variance.
    
    Args:
        X (pd.DataFrame or np.ndarray): Training features
        y (np.ndarray): Training labels  
        n_components (int): Number of discriminant components (max: n_classes-1)
    
    Returns:
        tuple: (lda_transformer, explained_variance_ratio)
    """
    lda = LinearDiscriminantAnalysis(n_components=n_components)
    lda.fit(X, y)
    evr = getattr(lda, 'explained_variance_ratio_', None)
    return lda, evr


def lda_feature_reduction(X, train_data, n_components=1, labels_key=LABELS_KEY, y=None):
    """
    Apply LDA transformation for supervised feature reduction.
    
    Fits LDA on training data and transforms provided features. For binary
    classification, only 1 discriminant component can be extracted.
    
    Args:
        X (pd.DataFrame): Features to transform
        train_data (pd.DataFrame): Training data for fitting LDA
        n_components (int): Number of components (1 for binary classification)
        labels_key (str): Label column name
        y (np.ndarray, optional): Training labels (extracted if None)
    
    Returns:
        tuple: (X_lda, explained_variance_ratio, lda_transformer)
            - X_lda: Transformed features in LDA space
            - explained_variance_ratio: Variance explained
            - lda_transformer: Fitted LDA object
    """
    # Extract training features and labels
    X_train = train_data.drop(columns=[labels_key]).astype(float)
    y_train = train_data[labels_key].to_numpy()
    
    # Fit LDA on training data
    lda, explained_var_ratio = fit_lda_reference(X_train, y_train, n_components=n_components)
    
    # Transform provided features
    X_lda = lda.transform(X)
    
    return X_lda, explained_var_ratio, lda

In [97]:
# Apply LDA to training data
lda_comp, explained_var_ratio, lda = lda_feature_reduction(
    train_data.drop(columns=[LABELS_KEY]),
    train_data
)

In [98]:
# Visualize LDA projection (1D)
fig = px.scatter(
    x=lda_comp[:, 0],
    y=[0] * lda_comp.shape[0],
    color=train_data[LABELS_KEY].astype(str),
    labels={'x': 'LDA Component', 'y': ''}
)
fig.update_yaxes(showticklabels=False)
fig.show()

### Classifiers

In [205]:
def calculate_metrics(y_true, y_pred, pos_label=2, neg_label=1): 
    """
    Calculate comprehensive classification performance metrics from confusion matrix.
    
    This function computes five essential metrics for binary classification evaluation:
    accuracy, sensitivity (recall), specificity, precision, and F1-score. All metrics
    are derived from the confusion matrix components (TP, TN, FP, FN).
    
    Metrics Definitions:
    - Accuracy: (TP + TN) / (TP + TN + FP + FN) - Overall correct predictions
    - Sensitivity (Recall): TP / (TP + FN) - True Positive Rate, ability to detect positives
    - Specificity: TN / (TN + FP) - True Negative Rate, ability to detect negatives
    - Precision: TP / (TP + FP) - Positive Predictive Value
    - F1-Score: 2 * (Precision * Recall) / (Precision + Recall) - Harmonic mean
    
    Args:
        y_true (array-like): True class labels, shape (n_samples,)
                            Values should be pos_label or neg_label
        y_pred (array-like): Predicted class labels, shape (n_samples,)
                            Values should be pos_label or neg_label
        pos_label (int): Label for positive class (breast cancer patients).
                        Default: 2
        neg_label (int): Label for negative class (healthy controls).
                        Default: 1
    
    Returns:
        tuple: (accuracy, sensitivity, specificity, precision, f1_score)
            All values are floats in range [0.0, 1.0]
            - accuracy (float): Overall classification accuracy
            - sensitivity (float): True positive rate (recall)

            - specificity (float): True negative rate    return accuracy, sensitivity, specificity, precision, f1

            - precision (float): Positive predictive value    

            - f1_score (float): Harmonic mean of precision and recall    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    """    
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    # Get confusion matrix components
    ixPos = np.where(y_true == pos_label)[0]
    ixNeg = np.where(y_true == neg_label)[0]
    
    TP = int(np.sum(y_pred[ixPos] == y_true[ixPos])) if ixPos.size > 0 else 0
    FN = int(np.sum(y_pred[ixPos] != y_true[ixPos])) if ixPos.size > 0 else 0
    TN = int(np.sum(y_pred[ixNeg] == y_true[ixNeg])) if ixNeg.size > 0 else 0
    FP = int(np.sum(y_pred[ixNeg] != y_true[ixNeg])) if ixNeg.size > 0 else 0
    
    # Calculate metrics
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0 # Recall = Sensitivity
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    
    return accuracy, sensitivity, specificity, precision, f1

In [100]:
def calculate_metrics_with_youden(y_true, y_probs, pos_label=2, neg_label=1):
    '''
    Calculate classification metrics using Youden's J statistic to find optimal threshold.
    Args:
        y_true (array-like): True class labels, shape (n_samples,)
                            Values should be pos_label or neg_label
        y_probs (array-like): Predicted probabilities for positive class, shape (n_samples,)
        pos_label (int): Label for positive class (breast cancer patients).
                        Default: 2
        neg_label (int): Label for negative class (healthy controls).
                        Default: 1
    '''
    y_true = np.asarray(y_true)
    y_probs = np.asarray(y_probs)
    
    fpr, tpr, thresholds = roc_curve(y_true, y_probs, pos_label=pos_label)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    
    y_pred = np.where(y_probs >= optimal_threshold, pos_label, neg_label)
    
    return calculate_metrics(y_true, y_pred, pos_label, neg_label)

#### Minimum Distance Classifiers

In [101]:
def calculate_youden_metrics(y_true, decision_scores, pos_label=2, neg_label=1):
    '''
    Calculate metrics at the optimal threshold using Youden Index.
    
    Args:
        y_true: true labels
        decision_scores: decision function scores or probabilities
        pos_label: positive class label (default: 2)
        neg_label: negative class label (default: 1)
    
    Returns:
        dict with optimal_threshold, sensitivity, specificity, precision, f1, and youden_index value
    '''
    y_true = np.asarray(y_true)
    decision_scores = np.asarray(decision_scores)
    
    fpr, tpr, thresholds = roc_curve(y_true, decision_scores, pos_label=pos_label)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    optimal_youden = youden_index[optimal_idx]
    
    # Create predictions using optimal threshold
    y_pred = np.where(decision_scores >= optimal_threshold, pos_label, neg_label)
    
    # Calculate metrics at optimal threshold
    accuracy, sensitivity, specificity, precision, f1 = calculate_metrics(y_true, y_pred, pos_label, neg_label)
    
    return {
        'optimal_threshold': float(optimal_threshold),
        'youden_index': float(optimal_youden),
        'sensitivity_youden': sensitivity,
        'specificity_youden': specificity,
        'precision_youden': precision,
        'f1_youden': f1,
        'accuracy_youden': accuracy
    }


In [102]:
class minimum_distance_classifier:
    """
    Minimum Distance Classifier using Euclidean or Mahalanobis distance metrics.
    
    This classifier assigns samples to the nearest class centroid (mean) using either
    Euclidean (assumes independent features) or Mahalanobis (accounts for correlations)
    distance. It implements discriminant functions for efficient computation.
    
    Mathematical Foundation:
    - Euclidean: g_i(x) = x^T μ_i - 0.5 * μ_i^T μ_i
    - Mahalanobis: g_i(x) = x^T Σ^(-1) μ_i - 0.5 * μ_i^T Σ^(-1) μ_i
    where μ_i is class centroid and Σ is covariance matrix
    
    Classification Rule: Assign x to class i if g_i(x) > g_j(x) for all j ≠ i
    """
    
    def __init__(self, pos_label=2, neg_label=1, dist='Euclidean'):
        """
        Initialize Minimum Distance Classifier.
        
        Args:
            pos_label (int): Label for positive class. Default: 2
            neg_label (int): Label for negative class. Default: 1
            dist (str): Distance metric. Options: 'Euclidean' or 'Mahalanobis'
                       Default: 'Euclidean'
        """
        self.centroids = {}
        self.pos_label = pos_label
        self.neg_label = neg_label
        self.dist = dist
        self.inv_cov = None

    def fit(self, X, y):
        """
        Fit the classifier by computing class centroids and covariance matrix.
        
        For Euclidean distance: Only computes class means (centroids)
        For Mahalanobis distance: Computes centroids + pooled covariance matrix inverse
        
        Args:
            X (np.ndarray): Training features, shape (n_samples, n_features)
            y (np.ndarray): Training labels, shape (n_samples,)
        
        Returns:
            self: Fitted classifier instance
        """
        X = np.asarray(X)
        y_arr = np.asarray(y)
        
        # Compute class centroids (mean vectors)
        self.centroids[self.pos_label] = np.mean(X[y_arr == self.pos_label], axis=0)
        self.centroids[self.neg_label] = np.mean(X[y_arr == self.neg_label], axis=0)

        # For Mahalanobis: compute pooled covariance matrix inverse
        if self.dist == 'Mahalanobis':
            cov = np.cov(X, rowvar=False)
            cov = np.atleast_2d(cov)
            cov = cov + 1e-8 * np.eye(cov.shape[0])  # Regularization
            self.inv_cov = np.linalg.pinv(cov)
        
        return self

    def g(self, X, label):
        """
        Compute discriminant function for a specific class.
        
        The discriminant function represents the "score" for assigning samples to a class.
        Higher score indicates higher confidence that sample belongs to that class.
        
        Args:
            X (np.ndarray): Samples to score, shape (n_samples, n_features) or (n_features,)
            label (int): Class label to compute discriminant for
        
        Returns:
            np.ndarray: Discriminant scores, shape (n_samples,)
        """
        centroid = self.centroids[label]
        X = np.asarray(X)

        if X.ndim == 1:
            X = X.reshape(1, -1)

        if self.dist == 'Euclidean':
            return X @ centroid - 0.5 * float(centroid @ centroid)
        elif self.dist == 'Mahalanobis':
            v = self.inv_cov @ centroid
            return X @ v - 0.5 * float(centroid @ v)

    def decision_function(self, X):
        """
        Compute decision scores for binary classification.
        
        Decision score is the difference between positive and negative discriminants:
        score = g_positive(x) - g_negative(x)
        
        Positive scores favor positive class, negative scores favor negative class.
        
        Args:
            X (np.ndarray): Samples, shape (n_samples, n_features)
        
        Returns:
            np.ndarray: Decision scores, shape (n_samples,)
                       score > 0: predicted as positive class
                       score < 0: predicted as negative class
        """
        X = np.asarray(X)
        g_pos = self.g(X, self.pos_label)
        g_neg = self.g(X, self.neg_label)
        return g_pos - g_neg

    def predict(self, X):
        """
        Predict class labels for samples.
        
        Assigns each sample to the class with highest discriminant score.
        Equivalent to choosing the nearest class centroid.
        
        Args:
            X (np.ndarray): Samples, shape (n_samples, n_features)
        
        Returns:
            np.ndarray: Predicted labels, shape (n_samples,)
                       Values are pos_label or neg_label
        """
        scores = self.decision_function(X)
        return np.where(scores > 0, self.pos_label, self.neg_label)

def run_min_dist_classifier(full_data, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, dist='Euclidean', n_times=150, pos_label=2, neg_label=1):
        """
        Execute Monte Carlo Cross-Validation for Minimum Distance Classifier.
        
        This function implements Repeated Sub-Sampling validation (Monte Carlo CV) to obtain
        robust performance estimates. It repeatedly splits data, trains classifier, evaluates
        on test set, and aggregates results across all iterations.
        
        Monte Carlo Cross-Validation Process:
        1. For each iteration (n_times = 150):
           a. Generate random train/test split with normalization
           b. Fit minimum distance classifier on training data
           c. Predict on test set
           d. Calculate all metrics (accuracy, sensitivity, specificity, precision, F1, ROC-AUC)
        2. Aggregate results: compute mean and standard deviation for each metric
        3. Return comprehensive performance summary
        
        Why Monte Carlo CV?
        - More robust than single train/test split (reduces variance)
        - Provides confidence intervals via standard deviations
        - Better estimate of generalization performance
        - Suitable for small datasets (116 samples)
        
        Args:
            full_data (pd.DataFrame): Complete dataset with features and labels
            labels_key (str): Column name for class labels. Default: 'Classification'
            train_size (float): Training set proportion. Default: 0.7 (70%)
            test_size (float): Testing set proportion. Default: 0.3 (30%)
            dist (str): Distance metric - 'Euclidean' or 'Mahalanobis'. Default: 'Euclidean'
            n_times (int): Number of Monte Carlo iterations. Default: 150
            pos_label (int): Positive class label. Default: 2
            neg_label (int): Negative class label. Default: 1
        
        Returns:
            dict: Performance metrics with mean and standard deviation:
                - 'accuracy_mean', 'accuracy_std': Overall classification accuracy
                - 'sensitivity_mean', 'sensitivity_std': True positive rate (recall)
                - 'specificity_mean', 'specificity_std': True negative rate
                - 'precision_mean', 'precision_std': Positive predictive value
                - 'f1_score_mean', 'f1_score_std': Harmonic mean of precision/recall
                - 'roc_auc_mean', 'roc_auc_std': Area under ROC curve
        
        Note:
            - Each iteration uses different random_state (RAND_SEED + i) for different splits
            - Standard deviation indicates stability/variability of performance
            - ROC-AUC uses decision_function scores (not hard predictions)
            - Youden Index metrics are computed but commented out (optional)
        """
        # Initialize lists to store metrics for each run
        accuracy_list = np.zeros(n_times)
        sensitivity_list = np.zeros(n_times)
        specificity_list = np.zeros(n_times)
        precision_list = np.zeros(n_times)
        f1_list = np.zeros(n_times)
        roc_auc_list = np.zeros(n_times)
        youden_list = np.zeros(n_times)
        sensitivity_youden_list = np.zeros(n_times)
        specificity_youden_list = np.zeros(n_times)
        precision_youden_list = np.zeros(n_times)
        f1_youden_list = np.zeros(n_times)
        
        # Monte Carlo Cross-Validation loop
        for i in range(n_times):
            # Generate train/test split with different random seed each iteration
            x_tr, y_tr, x_test, y_test = generate_testing_data(full_data, labels_key, random_state=RAND_SEED + i, train_size=train_size, test_size=test_size)
            
            # Train minimum distance classifier
            md_model = minimum_distance_classifier(dist=dist, pos_label=pos_label, neg_label=neg_label)
            md_model.fit(x_tr, y_tr)
            
            # Make predictions
            y_pred = md_model.predict(x_test)
        
            # Calculate classification metrics
            accuracy_list[i], sensitivity_list[i], specificity_list[i], precision_list[i], f1_list[i] = calculate_metrics(y_test, y_pred)

            # Calculate ROC-AUC using decision function scores
            decision_scores = md_model.decision_function(x_test)
            roc_auc_list[i] = roc_auc_score(y_test, decision_scores)
            
            # Calculate Youden Index metrics (optional - currently commented out in return)
            youden_metrics = calculate_youden_metrics(y_test, decision_scores, pos_label, neg_label)
            youden_list[i] = youden_metrics['youden_index']
            sensitivity_youden_list[i] = youden_metrics['sensitivity_youden']
            specificity_youden_list[i] = youden_metrics['specificity_youden']
            precision_youden_list[i] = youden_metrics['precision_youden']
            f1_youden_list[i] = youden_metrics['f1_youden']
        
        # Calculate means and standard deviations across all iterations
        metrics = {
            'accuracy_mean': np.mean(accuracy_list),
            'accuracy_std': np.std(accuracy_list),
            'sensitivity_mean': np.mean(sensitivity_list),
            'sensitivity_std': np.std(sensitivity_list),
            'specificity_mean': np.mean(specificity_list),
            'specificity_std': np.std(specificity_list),
            'precision_mean': np.mean(precision_list),
            'precision_std': np.std(precision_list),
            'f1_score_mean': np.mean(f1_list),
            'f1_score_std': np.std(f1_list),
            'roc_auc_mean': np.mean(roc_auc_list),
            'roc_auc_std': np.std(roc_auc_list),
            # Youden Index metrics (optional - uncomment if needed)
            # 'youden_index_mean': np.mean(youden_list),
            # 'youden_index_std': np.std(youden_list),
            # 'sensitivity_youden_mean': np.mean(sensitivity_youden_list),
            # 'sensitivity_youden_std': np.std(sensitivity_youden_list),
            # 'specificity_youden_mean': np.mean(specificity_youden_list),
            # 'specificity_youden_std': np.std(sensitivity_youden_list),
            # 'precision_youden_mean': np.mean(precision_youden_list),
            # 'precision_youden_std': np.std(precision_youden_list),
            # 'f1_youden_mean': np.mean(f1_youden_list),
            # 'f1_youden_std': np.std(f1_youden_list)
        }
        
        return metrics

In [103]:
# Evaluate Minimum Distance Classifier on full dataset
print('=== Euclidean Distance ===')
euc_results = run_min_dist_classifier(blood_data, dist='Euclidean')
print(euc_results)

print('\n=== Mahalanobis Distance ===')
mah_results = run_min_dist_classifier(blood_data, dist='Mahalanobis')
print(mah_results)

--- Euclidean Distance ---
{'accuracy_mean': 0.6485714285714285, 'accuracy_std': 0.07248739041176394, 'sensitivity_mean': 0.5177875198138088, 'sensitivity_std': 0.11204284642959501, 'specificity_mean': 0.8112019050494281, 'specificity_std': 0.10124377003799936, 'precision_mean': 0.7733407776434091, 'precision_std': 0.11360601399199832, 'f1_score_mean': 0.6115908528049107, 'f1_score_std': 0.0955472670582986, 'roc_auc_mean': 0.7500656192736634, 'roc_auc_std': 0.08074831615925679}
--- Mahalanobis Distance ---
{'accuracy_mean': 0.6965714285714284, 'accuracy_std': 0.07433478232930786, 'sensitivity_mean': 0.6478087680731845, 'sensitivity_std': 0.10676703787811061, 'specificity_mean': 0.7608293816892888, 'specificity_std': 0.12442002796269538, 'precision_mean': 0.7708305334156726, 'precision_std': 0.11469849873270394, 'f1_score_mean': 0.6965519283885697, 'f1_score_std': 0.08529197671080259, 'roc_auc_mean': 0.7617885174489305, 'roc_auc_std': 0.07700142505773419}


In [104]:
# Run Minimum Distance Classifier on PCA reduced data
def run_min_dist_classifier_pca(blood_data, n_components, random_state=RAND_SEED, dist='Euclidean', labels_key=LABELS_KEY, n_times=150):
    """
    Evaluate minimum distance classifier with PCA dimensionality reduction using Monte Carlo CV.
    
    Applies PCA to reduce features to n_components, then trains and tests minimum distance
    classifier multiple times with different random splits to estimate performance.
    
    Args:
        blood_data: Full dataset with features and labels
        n_components: Number of principal components to keep
        random_state: Random seed for reproducibility
        dist: Distance metric ('Euclidean' or 'Mahalanobis')
        labels_key: Name of the label column
        n_times: Number of Monte Carlo iterations
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    feat_cols = [c for c in blood_data.columns if c != labels_key]
    
    # Initialize arrays to store metrics for each iteration
    euc_accuracies = np.zeros(n_times)
    euc_sensitivities = np.zeros(n_times)
    euc_specificities = np.zeros(n_times)
    euc_precisions = np.zeros(n_times)
    euc_f1s = np.zeros(n_times)
    euc_roc_aucs = np.zeros(n_times)
    
    for i in range(n_times):
        # Generate new train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(blood_data, random_state=random_state + i)
        
        # Apply PCA reduction
        pca = PCA(n_components=n_components)
        pca.fit(pd.DataFrame(x_tr, columns=feat_cols))
        pca_comp_tr = pca.transform(pd.DataFrame(x_tr, columns=feat_cols))
        pca_comp_test = pca.transform(pd.DataFrame(x_test, columns=feat_cols))
        
        # Train and predict with minimum distance classifier
        md_model = minimum_distance_classifier(dist=dist)
        md_model.fit(pca_comp_tr, y_tr)
        y_pred = md_model.predict(pca_comp_test)
        
        # Calculate metrics
        euc_accuracies[i], euc_sensitivities[i], euc_specificities[i], euc_precisions[i], euc_f1s[i] = calculate_metrics(y_test, y_pred)
        
        # Calculate ROC-AUC using decision function
        decision_scores = md_model.decision_function(pca_comp_test)
        euc_roc_aucs[i] = roc_auc_score(y_test, decision_scores)
    
    # Return mean and standard deviation for each metric
    metrics = {
        'accuracy_mean': np.mean(euc_accuracies),
        'accuracy_std': np.std(euc_accuracies),
        'sensitivity_mean': np.mean(euc_sensitivities),
        'sensitivity_std': np.std(euc_sensitivities),
        'specificity_mean': np.mean(euc_specificities),
        'specificity_std': np.std(euc_specificities),
        'precision_mean': np.mean(euc_precisions),
        'precision_std': np.std(euc_precisions),
        'f1_score_mean': np.mean(euc_f1s),
        'f1_score_std': np.std(euc_f1s),
        'roc_auc_mean': np.mean(euc_roc_aucs),
        'roc_auc_std': np.std(euc_roc_aucs)
    }
    return metrics


In [129]:
print('--- Minimum Distance Classifier on PCA reduced data ---')
print('---Euclidean Distance---')
for i in range(1, blood_data.shape[1]):
    min_dist_euc_lda_results = run_min_dist_classifier_pca(blood_data, n_components=i, dist='Euclidean')
    print(f'--- PCA with {i} components ---')
    print(min_dist_euc_lda_results)

print('---Mahalanobis Distance---')
for i in range(1, blood_data.shape[1]):
    min_dist_mah_lda_results = run_min_dist_classifier_pca(blood_data, n_components=i, dist='Mahalanobis')
    print(f'--- PCA with {i} components ---')
    print(min_dist_mah_lda_results)

print('--- Minimum Distance Classifier with Kruskal-Wallis Features ---')
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data_subset = blood_data[selected_features]
    print(f'--- Top {i} Kruskal features ---')
    print(run_min_dist_classifier(data_subset, dist='Mahalanobis'))

--- Minimum Distance Classifier on PCA reduced data ---
---Euclidean Distance---
--- PCA with 1 components ---
{'accuracy_mean': 0.576952380952381, 'accuracy_std': 0.07027229191749669, 'sensitivity_mean': 0.4961402905174274, 'sensitivity_std': 0.11093142366177992, 'specificity_mean': 0.6772072980339233, 'specificity_std': 0.11137598195006793, 'precision_mean': 0.6551675803842987, 'precision_std': 0.10196284726217647, 'f1_score_mean': 0.5565358816761798, 'f1_score_std': 0.08967597300889674, 'roc_auc_mean': 0.6468422294957324, 'roc_auc_std': 0.08250354009360733}
--- PCA with 2 components ---
{'accuracy_mean': 0.6081904761904762, 'accuracy_std': 0.07070772777030748, 'sensitivity_mean': 0.48802789573737215, 'sensitivity_std': 0.11154996373336036, 'specificity_mean': 0.7568555504522997, 'specificity_std': 0.11427134110745597, 'precision_mean': 0.7161559022333016, 'precision_std': 0.11056102491272464, 'f1_score_mean': 0.5709070617942551, 'f1_score_std': 0.09294750794132972, 'roc_auc_mean': 0

In [106]:
# Apply LDA to complete dataset (using train_data as reference)
lda_comp, _, _ = lda_feature_reduction(blood_data.drop(columns=[LABELS_KEY]), train_data)
lda_data = pd.DataFrame(lda_comp)
lda_data[LABELS_KEY] = blood_data[LABELS_KEY].to_numpy()

def run_min_dist_classifier_lda(full_data, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150, pos_label=2, neg_label=1):
    """
    Evaluate minimum distance classifier with LDA dimensionality reduction using Monte Carlo CV.
    
    Applies LDA to reduce features to 1D (binary classification), then trains and tests
    minimum distance classifier multiple times with different random splits.
    
    Args:
        full_data: Complete dataset with features and labels
        labels_key: Name of the label column
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        n_times: Number of Monte Carlo iterations
        pos_label: Label for positive class
        neg_label: Label for negative class
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    euc_accuracies = np.zeros(n_times)
    euc_sensitivities = np.zeros(n_times)
    euc_specificities = np.zeros(n_times)
    euc_precisions = np.zeros(n_times)
    euc_f1s = np.zeros(n_times)
    euc_roc_aucs = np.zeros(n_times)

    for i in range(n_times):
        # Generate train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(full_data, labels_key, random_state=RAND_SEED + i, train_size=train_size, test_size=test_size)
        
        # Apply LDA transformation
        lda = LinearDiscriminantAnalysis(n_components=1)
        lda.fit(x_tr, y_tr)
        x_tr_lda = lda.transform(x_tr)
        x_test_lda = lda.transform(x_test)

        # Train and predict
        md_model = minimum_distance_classifier(dist='Euclidean', pos_label=pos_label, neg_label=neg_label)
        md_model.fit(x_tr_lda, y_tr)
        y_pred = md_model.predict(x_test_lda)

        # Calculate metrics
        euc_accuracies[i], euc_sensitivities[i], euc_specificities[i], euc_precisions[i], euc_f1s[i] = calculate_metrics(y_test, y_pred)
        decision_scores = md_model.decision_function(x_test_lda)
        euc_roc_aucs[i] = roc_auc_score(y_test, decision_scores)

    return {
        'accuracy_mean': np.mean(euc_accuracies),
        'accuracy_std': np.std(euc_accuracies),
        'sensitivity_mean': np.mean(euc_sensitivities),
        'sensitivity_std': np.std(euc_sensitivities),
        'specificity_mean': np.mean(euc_specificities),
        'specificity_std': np.std(euc_specificities),
        'precision_mean': np.mean(euc_precisions),
        'precision_std': np.std(euc_precisions),
        'f1_score_mean': np.mean(euc_f1s),
        'f1_score_std': np.std(euc_f1s),
        'roc_auc_mean': np.mean(euc_roc_aucs),
        'roc_auc_std': np.std(euc_roc_aucs)
    }


In [107]:
print('--- Minimum Distance Classifier on LDA reduced data ---')
min_dist_euc_lda_results = run_min_dist_classifier_lda(lda_data)
print(min_dist_euc_lda_results)
# For this classifier, Mahalanobis distance does not make sense with LDA reduced data, it's going to return the same results as Euclidean

--- Minimum Distance Classifier on LDA reduced data ---
{'accuracy_mean': 0.603047619047619, 'accuracy_std': 0.08307994128543264, 'sensitivity_mean': 0.49676028362114644, 'sensitivity_std': 0.13148579423933085, 'specificity_mean': 0.7330412952773634, 'specificity_std': 0.11127132431989098, 'precision_mean': 0.6949109274511752, 'precision_std': 0.11640172480131154, 'f1_score_mean': 0.5699283851940032, 'f1_score_std': 0.11191374597054018, 'roc_auc_mean': 0.6974235347851407, 'roc_auc_std': 0.08298884137492579}


In [108]:
class fisher_lda_classifier:
    def __init__(self, pos_label=2, neg_label=1):
        self.w = None
        self.b = None
        self.pos_label = pos_label
        self.neg_label = neg_label
        self.mu_pos = None
        self.mu_neg = None

    def fit(self, X, y):
        """Fit Fisher LDA with a fixed notion of positive/negative labels to avoid sign flips."""
        X = np.asarray(X)
        y = np.asarray(y)

        X_pos = X[y == self.pos_label]
        X_neg = X[y == self.neg_label]
        self.mu_pos = X_pos.mean(axis=0)
        self.mu_neg = X_neg.mean(axis=0)

        S_pos = (X_pos - self.mu_pos).T @ (X_pos - self.mu_pos)
        S_neg = (X_neg - self.mu_neg).T @ (X_neg - self.mu_neg)
        Sw = S_pos + S_neg

        # Direction points from negative to positive class
        self.w = np.linalg.pinv(Sw) @ (self.mu_pos - self.mu_neg)
        self.b = -0.5 * (self.mu_pos + self.mu_neg) @ self.w
        return self

    def predict(self, X):
        X = np.asarray(X)
        scores = self.decision_function(X)
        return np.where(scores >= 0, self.pos_label, self.neg_label)

    def decision_function(self, X):
        if self.w is None or self.b is None:
            raise ValueError("Classifier must be fitted before prediction")
        X = np.asarray(X)
        return X @ self.w + self.b

def run_fisher_lda_classifier(full_data, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150, pos_label=2, neg_label=1):
    accuracy_list = np.zeros(n_times)
    sensitivity_list = np.zeros(n_times)
    specificity_list = np.zeros(n_times)
    precision_list = np.zeros(n_times)
    f1_list = np.zeros(n_times)
    roc_auc_list = np.zeros(n_times)
    youden_list = np.zeros(n_times)
    sensitivity_youden_list = np.zeros(n_times)
    specificity_youden_list = np.zeros(n_times)
    precision_youden_list = np.zeros(n_times)
    f1_youden_list = np.zeros(n_times)
    
    for i in range(n_times):
        x_tr, y_tr, x_test, y_test = generate_testing_data(full_data, labels_key, random_state=RAND_SEED + i, train_size=train_size, test_size=test_size)
        fisher_model = fisher_lda_classifier(pos_label=pos_label, neg_label=neg_label)
        fisher_model.fit(x_tr, y_tr)
        y_pred = fisher_model.predict(x_test)
        
        # Calculate metrics
        accuracy_list[i], sensitivity_list[i], specificity_list[i], precision_list[i], f1_list[i] = calculate_metrics(y_test, y_pred, pos_label, neg_label)
        
        # Calculate ROC-AUC using decision function
        decision_scores = fisher_model.decision_function(x_test)
        roc_auc_list[i] = roc_auc_score(y_test, decision_scores)
        
        # Calculate Youden Index metrics
        youden_metrics = calculate_youden_metrics(y_test, decision_scores, pos_label, neg_label)
        youden_list[i] = youden_metrics['youden_index']
        sensitivity_youden_list[i] = youden_metrics['sensitivity_youden']
        specificity_youden_list[i] = youden_metrics['specificity_youden']
        precision_youden_list[i] = youden_metrics['precision_youden']
        f1_youden_list[i] = youden_metrics['f1_youden']
    
    # Calculate means and standard deviations
    metrics = {
        'accuracy_mean': np.mean(accuracy_list),
        'accuracy_std': np.std(accuracy_list),
        'sensitivity_mean': np.mean(sensitivity_list),
        'sensitivity_std': np.std(sensitivity_list),
        'specificity_mean': np.mean(specificity_list),
        'specificity_std': np.std(specificity_list),
        'precision_mean': np.mean(precision_list),
        'precision_std': np.std(precision_list),
        'f1_score_mean': np.mean(f1_list),
        'f1_score_std': np.std(f1_list),
        'roc_auc_mean': np.mean(roc_auc_list),
        'roc_auc_std': np.std(roc_auc_list),
        # 'youden_index_mean': np.mean(youden_list),
        # 'youden_index_std': np.std(youden_list),
        # 'sensitivity_youden_mean': np.mean(sensitivity_youden_list),
        # 'sensitivity_youden_std': np.std(sensitivity_youden_list),
        # 'specificity_youden_mean': np.mean(specificity_youden_list),
        # 'specificity_youden_std': np.std(sensitivity_youden_list),
        # 'precision_youden_mean': np.mean(precision_youden_list),
        # 'precision_youden_std': np.std(precision_youden_list),
        # 'f1_youden_mean': np.mean(f1_youden_list),
        # 'f1_youden_std': np.std(f1_youden_list)
    }
    return metrics

In [109]:
def run_fisher_lda_classifier_reduced_data(full_data, model={'reduction':'PCA', 'n_components':3}, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150, pos_label=2, neg_label=1):
    """
    Evaluate Fisher LDA classifier with dimensionality reduction using Monte Carlo CV.
    
    Applies either PCA or LDA for dimensionality reduction, then trains and tests
    Fisher LDA classifier multiple times with different random splits.
    
    Args:
        full_data: Complete dataset with features and labels
        model: Dictionary specifying reduction method and number of components
               e.g., {'reduction':'PCA', 'n_components':3}
        labels_key: Name of the label column
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        n_times: Number of Monte Carlo iterations
        pos_label: Label for positive class
        neg_label: Label for negative class
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    n_components = model.get('n_components', None)
    if n_components is None:
        raise ValueError("n_components must be specified in the model dictionary")
    method = model.get('reduction', 'PCA')

    # Initialize metric arrays
    accuracies = np.zeros(n_times)
    sensitivities = np.zeros(n_times)
    specificities = np.zeros(n_times)
    precisions = np.zeros(n_times)
    f1s = np.zeros(n_times)
    roc_aucs = np.zeros(n_times)
    
    for i in range(n_times):
        # Generate train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(full_data, labels_key, random_state=RAND_SEED + i, train_size=train_size, test_size=test_size)
        
        # Apply dimensionality reduction
        if method == 'PCA':
            pca = fit_pca_reference(pd.DataFrame(x_tr, columns=[c for c in full_data.columns if c != labels_key]), n_components=n_components)
            x_tr_reduced = pca.transform(pd.DataFrame(x_tr, columns=[c for c in full_data.columns if c != labels_key]))
            x_test_reduced = pca.transform(pd.DataFrame(x_test, columns=[c for c in full_data.columns if c != labels_key]))
        elif method == 'LDA':
            lda = LinearDiscriminantAnalysis(n_components=n_components)
            lda.fit(x_tr, y_tr)
            x_tr_reduced = lda.transform(x_tr)
            x_test_reduced = lda.transform(x_test)
        else:
            raise ValueError("Unknown reduction method. Use 'PCA' or 'LDA'.")
        
        # Train Fisher LDA classifier on reduced data
        fisher_model = fisher_lda_classifier(pos_label=pos_label, neg_label=neg_label)
        fisher_model.fit(x_tr_reduced, y_tr)
        y_pred = fisher_model.predict(x_test_reduced)

        # Calculate metrics
        accuracies[i], sensitivities[i], specificities[i], precisions[i], f1s[i] = calculate_metrics(y_test, y_pred)
        decision_scores = fisher_model.decision_function(x_test_reduced)
        roc_aucs[i] = roc_auc_score(y_test, decision_scores)
    
    # Return mean and std for all metrics
    results = {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1s),
        'f1_score_std': np.std(f1s),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }
    return results


In [110]:
# Aplicar PCA ao dataset completo para diferentes números de componentes
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]

print('--- Fisher LDA Classifier with PCA Components (Monte Carlo CV) ---')
print()

for n_comp in range(1, n_features + 1):
    print(f'--- {n_comp} PCA components ---')
    fisher_pca_results = run_fisher_lda_classifier_reduced_data(
        blood_data,
        model={'reduction':'PCA', 'n_components':n_comp},
        )
    print(fisher_pca_results)

print('--- Fisher LDA Classifier with Kruskal-Wallis Features ---')
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data_subset = blood_data[selected_features]
    print(f'--- Top {i} Kruskal features ---')
    print(run_fisher_lda_classifier(data_subset))

--- Fisher LDA Classifier with PCA Components (Monte Carlo CV) ---

--- 1 PCA components ---
{'accuracy_mean': 0.576952380952381, 'accuracy_std': 0.07027229191749669, 'sensitivity_mean': 0.4961402905174274, 'sensitivity_std': 0.11093142366177992, 'specificity_mean': 0.6772072980339233, 'specificity_std': 0.11137598195006793, 'precision_mean': 0.6551675803842987, 'precision_std': 0.10196284726217647, 'f1_score_mean': 0.5565358816761798, 'f1_score_std': 0.08967597300889674, 'roc_auc_mean': 0.6468422294957324, 'roc_auc_std': 0.08250354009360733}
--- 2 PCA components ---
{'accuracy_mean': 0.5982857142857142, 'accuracy_std': 0.07560945123227825, 'sensitivity_mean': 0.47638080390143256, 'sensitivity_std': 0.11956782113387125, 'specificity_mean': 0.7487485753553865, 'specificity_std': 0.11285811311663385, 'precision_mean': 0.7026641451980663, 'precision_std': 0.1200779877141427, 'f1_score_mean': 0.557794584285415, 'f1_score_std': 0.10376947291930631, 'roc_auc_mean': 0.6852629738841461, 'roc_a

In [111]:
# Aplicar LDA ao dataset completo (usando train_data como referência)
lda_comp, _, _ = lda_feature_reduction(blood_data.drop(columns=[LABELS_KEY]), train_data)
lda_data = pd.DataFrame(lda_comp)
lda_data[LABELS_KEY] = blood_data[LABELS_KEY].to_numpy()

print('--- Fisher LDA Classifier with LDA Components (Monte Carlo CV) ---')
fisher_results = run_fisher_lda_classifier_reduced_data(
    lda_data,
    model={'reduction':'LDA', 'n_components':1},
    )
print(fisher_results)

--- Fisher LDA Classifier with LDA Components (Monte Carlo CV) ---
{'accuracy_mean': 0.603047619047619, 'accuracy_std': 0.08307994128543264, 'sensitivity_mean': 0.49676028362114644, 'sensitivity_std': 0.13148579423933085, 'specificity_mean': 0.7330412952773634, 'specificity_std': 0.11127132431989098, 'precision_mean': 0.6949109274511752, 'precision_std': 0.11640172480131154, 'f1_score_mean': 0.5699283851940032, 'f1_score_std': 0.11191374597054018, 'roc_auc_mean': 0.6974235347851407, 'roc_auc_std': 0.08298884137492579}


## Final Checkpoint

#### Bayes Classifier

In [112]:
# Import Gaussian Mixture Models for Bayes classifier
from sklearn import mixture

In [130]:
x_tr = train_data.drop(columns=[LABELS_KEY]).to_numpy()
y_tr = train_data[LABELS_KEY].to_numpy()
x_test = test_data.drop(columns=[LABELS_KEY]).to_numpy()
y_test = test_data[LABELS_KEY].to_numpy()

In [131]:
def fit_bayes(x_tr, y_tr):
    """
    Fit Gaussian Naive Bayes classifier using Maximum Likelihood Estimation.
    
    Estimates class-conditional Gaussian distributions (mean and covariance)
    for each class. Used for generative classification via Bayes' theorem.
    
    Args:
        x_tr (np.ndarray): Training features, shape (n_samples, n_features)
        y_tr (np.ndarray): Training labels, shape (n_samples,)
    
    Returns:
        dict: Model parameters for each class
            {class_label: {
                'mean': mean vector,
                'cov': covariance matrix,
                'prior': P(class)
            }}
    """
    classes = np.unique(y_tr)
    models = {}
    
    for c in classes:
        # Extract samples from this class
        x_c = x_tr[y_tr == c]
        prior = len(x_c) / len(x_tr)  # P(class)
        
        # Fit Gaussian to class data
        clf = mixture.GaussianMixture(n_components=1)
        clf.fit(x_c)
        
        models[c] = {
            'mean': clf.means_.squeeze(),
            'cov': clf.covariances_[0],
            'prior': prior
        }
    
    return models

In [132]:
def pdf_gauss(X, mean, cov):
    """
    Compute multivariate Gaussian probability density function.
    
    Calculates P(x | class) - the likelihood of observing samples
    under a Gaussian distribution with given parameters.
    
    Formula: p(x) = (1/√((2π)^d|Σ|)) * exp(-0.5*(x-μ)^T Σ^(-1) (x-μ))
    
    Args:
        X (np.ndarray): Samples, shape (n_samples, n_features)
        mean (np.ndarray): Mean vector, shape (n_features,)
        cov (np.ndarray): Covariance matrix, shape (n_features, n_features)
    
    Returns:
        np.ndarray: Probability densities, shape (n_samples,)
    """
    dim = cov.shape[0]
    covInv = np.linalg.inv(cov)
    val = np.array([])
    
    for i in range(X.shape[0]):
        # Mahalanobis distance: (x-μ)^T Σ^(-1) (x-μ)
        dist = ((np.array([X[i, :] - mean])) @ covInv @ (np.array([X[i, :] - mean])).T).squeeze()
        
        # Multivariate Gaussian PDF
        multivariate_pdf = np.exp(-0.5 * dist) / ((2 * np.pi) ** (dim / 2) * np.linalg.det(cov) ** 0.5)
        val = np.append(val, multivariate_pdf)
    
    return np.array([val]).T.squeeze()

In [133]:
def use_bayes(models, X, return_proba=False):
    """
    Make predictions using Bayesian inference (MAP classification).
    
    Computes posterior probabilities P(class|x) using Bayes' theorem
    and assigns samples to class with highest posterior (MAP).
    
    Bayes' Theorem: P(class|x) ∝ P(x|class) * P(class)
    
    Args:
        models (dict): Fitted Bayes models from fit_bayes()
        X (np.ndarray): Samples, shape (n_samples, n_features)
        return_proba (bool): Return probabilities instead of predictions
    
    Returns:
        np.ndarray: Class labels or posterior probabilities
    """
    classes = list(models.keys())
    probs = np.zeros((X.shape[0], len(classes)))
    
    # Compute posterior probability for each class
    for i, c in enumerate(classes):
        mean = models[c]['mean']
        cov = models[c]['cov']
        prior = models[c]['prior']
        
        # Posterior ∝ Likelihood * Prior
        likelihood = pdf_gauss(X, mean, cov)
        probs[:, i] = likelihood * prior
    
    # Return normalized probabilities if requested
    if return_proba:
        denom = probs.sum(axis=1, keepdims=True)
        denom[denom == 0] = 1.0  # Avoid division by zero
        return probs / denom
    
    # Return class predictions (MAP classification)
    preds_idx = np.argmax(probs, axis=1)
    return np.array([classes[i] for i in preds_idx])

In [144]:
def bayes_classifier_full_evaluation(data, reduction_model=None, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150, pos_label=2, neg_label=1):
    """
    Evaluate Gaussian Bayes classifier using Monte Carlo cross-validation.
    
    Trains and tests Bayes classifier multiple times with different random splits,
    optionally applying dimensionality reduction (PCA or LDA) before classification.
    
    Args:
        data: Complete dataset with features and labels
        reduction_model: Optional dictionary specifying dimensionality reduction
                        e.g., {'reduction':'PCA', 'n_components':5}
                        If None, uses all features
        labels_key: Name of the label column
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        n_times: Number of Monte Carlo iterations
        pos_label: Label for positive class
        neg_label: Label for negative class
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    # Initialize metric arrays
    accuracies = np.zeros(n_times)
    sensitivities = np.zeros(n_times)
    specificities = np.zeros(n_times)
    precisions = np.zeros(n_times)
    f1_scores = np.zeros(n_times)
    roc_aucs = np.zeros(n_times)
    youden_list = np.zeros(n_times)
    sensitivity_youden_list = np.zeros(n_times)
    specificity_youden_list = np.zeros(n_times)
    precision_youden_list = np.zeros(n_times)
    f1_youden_list = np.zeros(n_times)
    
    for i in range(n_times):
        # Generate train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(data, random_state=RAND_SEED + i, labels_key=labels_key, train_size=train_size, test_size=test_size)

        # Apply dimensionality reduction if specified
        if reduction_model is not None:
            method = reduction_model.get('reduction', 'PCA')
            n_components = reduction_model.get('n_components', None)
            if method == 'PCA':
                pca = fit_pca_reference(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]), n_components=n_components)
                x_tr = pca.transform(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]))
                x_test = pca.transform(pd.DataFrame(x_test, columns=[c for c in data.columns if c != labels_key]))
            elif method == 'LDA':
                lda = LinearDiscriminantAnalysis(n_components=n_components)
                lda.fit(x_tr, y_tr)
                x_tr = lda.transform(x_tr)
                x_test = lda.transform(x_test)
            else:
                raise ValueError("Unknown reduction method. Use 'PCA' or 'LDA'.")

        # Train Bayes classifier and make predictions
        bayes_model = fit_bayes(x_tr, y_tr)
        pred_y = use_bayes(bayes_model, x_test)
        pred_proba = use_bayes(bayes_model, x_test, return_proba=True)
        
        # Calculate standard metrics
        accuracies[i], sensitivities[i], specificities[i], precisions[i], f1_scores[i] = calculate_metrics(y_test, pred_y, pos_label, neg_label)
        
        # Calculate ROC-AUC using probabilities for positive class
        classes = list(bayes_model.keys())
        pos_idx = list(classes).index(pos_label)
        decision_scores = pred_proba[:, pos_idx]
        roc_aucs[i] = roc_auc_score(y_test, decision_scores)
        
        # Calculate Youden Index metrics
        youden_metrics = calculate_youden_metrics(y_test, decision_scores, pos_label, neg_label)
        youden_list[i] = youden_metrics['youden_index']
        sensitivity_youden_list[i] = youden_metrics['sensitivity_youden']
        specificity_youden_list[i] = youden_metrics['specificity_youden']
        precision_youden_list[i] = youden_metrics['precision_youden']
        f1_youden_list[i] = youden_metrics['f1_youden']
    
    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1_scores),
        'f1_score_std': np.std(f1_scores),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }


In [135]:
# Apply Bayes Classifier on full dataset
print('--- Bayes Classifier on Full Dataset ---')
print(bayes_classifier_full_evaluation(blood_data))

--- Bayes Classifier on Full Dataset ---
{'accuracy_mean': 0.6563809523809524, 'accuracy_std': 0.07227661653019016, 'sensitivity_mean': 0.5367087982260481, 'sensitivity_std': 0.11960050887635203, 'specificity_mean': 0.808419261337218, 'specificity_std': 0.10451286472308884, 'precision_mean': 0.7788491400108642, 'precision_std': 0.10466179467628249, 'f1_score_mean': 0.6249256855189882, 'f1_score_std': 0.09198426492175349, 'roc_auc_mean': 0.7639895439223591, 'roc_auc_std': 0.06906541730497097}


In [140]:
# using only some of the ranked features
kruskal_ranking_bayes_results = []
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data = blood_data[selected_features]
    kruskal_ranking_bayes_results.append(bayes_classifier_full_evaluation(data))

for i, test in enumerate(kruskal_ranking_bayes_results):
    print('top ' + str(i + 1) + ' Kruskal features: ' + str(test))


top 1 Kruskal features: {'accuracy_mean': 0.6167619047619047, 'accuracy_std': 0.0706191600037638, 'sensitivity_mean': 0.4452537956940434, 'sensitivity_std': 0.1156342968344971, 'specificity_mean': 0.8341576664363042, 'specificity_std': 0.08540545161200122, 'precision_mean': 0.7687178599121323, 'precision_std': 0.11102035312548739, 'f1_score_mean': 0.5527888986252126, 'f1_score_std': 0.09684914502738336, 'roc_auc_mean': 0.7314748375411144, 'roc_auc_std': 0.06817028918274748}
top 2 Kruskal features: {'accuracy_mean': 0.6373333333333332, 'accuracy_std': 0.07813796740454274, 'sensitivity_mean': 0.4436388511542301, 'sensitivity_std': 0.11428230179666522, 'specificity_mean': 0.8763285402821005, 'specificity_std': 0.08950639283744603, 'precision_mean': 0.8224990477931654, 'precision_std': 0.11499407077514232, 'f1_score_mean': 0.5661035877717889, 'f1_score_std': 0.10535984649381722, 'roc_auc_mean': 0.7797072131419017, 'roc_auc_std': 0.06776452655315733}
top 3 Kruskal features: {'accuracy_mean'

In [142]:
# with PCA components

pca_bayes_results = []
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]
for i in range(1, n_features + 1):
    pca_bayes_results.append(
        bayes_classifier_full_evaluation(
            blood_data,
            reduction_model={'reduction':'PCA', 'n_components':i}
        )
    )

for i, test in enumerate(pca_bayes_results):
    print('top ' + str(i + 1) + ' PCA components: ' + str(test))

top 1 PCA components: {'accuracy_mean': 0.5521904761904761, 'accuracy_std': 0.06307293406317696, 'sensitivity_mean': 0.4155890576300255, 'sensitivity_std': 0.11772153951993346, 'specificity_mean': 0.735728852531484, 'specificity_std': 0.12551940800566183, 'precision_mean': 0.662679221845337, 'precision_std': 0.13149347530840808, 'f1_score_mean': 0.4958942032767645, 'f1_score_std': 0.09215296501989004, 'roc_auc_mean': 0.5830218092616737, 'roc_auc_std': 0.07996255173240753}
top 2 PCA components: {'accuracy_mean': 0.5893333333333334, 'accuracy_std': 0.07373766086868697, 'sensitivity_mean': 0.38902952334902796, 'sensitivity_std': 0.11605084023717585, 'specificity_mean': 0.8416815524803141, 'specificity_std': 0.09831141986997083, 'precision_mean': 0.7537001549648609, 'precision_std': 0.13869106107719037, 'f1_score_mean': 0.5012414642986391, 'f1_score_std': 0.10767143434900835, 'roc_auc_mean': 0.6421403159572848, 'roc_auc_std': 0.0884309095105696}
top 3 PCA components: {'accuracy_mean': 0.60

In [145]:
# with LDA component
print('--- Bayes Classifier on LDA reduced data ---')
lda_bayes_results = bayes_classifier_full_evaluation(
    lda_data,
    reduction_model={'reduction':'LDA', 'n_components':1}
)
print(lda_bayes_results)

--- Bayes Classifier on LDA reduced data ---
{'accuracy_mean': 0.5655238095238094, 'accuracy_std': 0.07357141316073194, 'sensitivity_mean': 0.3906648097419399, 'sensitivity_std': 0.12627474090889199, 'specificity_mean': 0.7874541465586358, 'specificity_std': 0.10869537406953984, 'precision_mean': 0.6937544150322789, 'precision_std': 0.13261333594222785, 'f1_score_mean': 0.4862139186526221, 'f1_score_std': 0.11270747108538902, 'roc_auc_mean': 0.6217281837381737, 'roc_auc_std': 0.09119420430018797}


## KNN Classifier

In [152]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def run_knn(x_tr, y_tr, x_test, y_test, k=5, weights='uniform', metric='minkowski', pos_label=2, neg_label=1):
    """
    Executa um treino e teste de KNN uma única vez com os parâmetros fornecidos.
    Args:
        
    """
    
    knn = KNeighborsClassifier(n_neighbors=k, weights=weights, metric=metric)
    
    knn.fit(x_tr, y_tr)

    y_pred = knn.predict(x_test)
    
    # metrics
    accuracy, sensitivity, specificity, precision, f1 = calculate_metrics(y_test, y_pred, pos_label, neg_label)
    
    # Get probability predictions for ROC-AUC
    y_proba = knn.predict_proba(x_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_proba)
    
    # Calculate Youden Index metrics
    # youden_metrics = calculate_youden_metrics(y_test, y_proba, pos_label, neg_label)
    
    return accuracy, sensitivity, specificity, precision, f1, roc_auc

In [150]:
def knn_c_full_test(data, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150, k=5, weights='uniform', metric='minkowski', pos_label=2, neg_label=1):
    """
    Evaluate K-Nearest Neighbors classifier using Monte Carlo cross-validation.
    
    Trains and tests KNN classifier multiple times with different random splits
    to estimate performance with specified hyperparameters.
    
    Args:
        data: Complete dataset with features and labels
        labels_key: Name of the label column
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        n_times: Number of Monte Carlo iterations
        k: Number of neighbors to consider
        weights: Weight function ('uniform' or 'distance')
        metric: Distance metric ('minkowski', 'euclidean', 'manhattan', etc.)
        pos_label: Label for positive class
        neg_label: Label for negative class
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    # Initialize metric arrays
    accuracies = np.zeros(n_times)
    sensitivities = np.zeros(n_times)
    specificities = np.zeros(n_times)
    precisions = np.zeros(n_times)
    f1_scores = np.zeros(n_times)
    roc_aucs = np.zeros(n_times)

    for i in range(n_times):
        # Generate train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(
            data,
            random_state=RAND_SEED + i,
            labels_key=labels_key,
            train_size=train_size,
            test_size=test_size
        )

        # Train and evaluate KNN
        accuracy, sensitivity, specificity, precision, f1, roc_auc = run_knn(
            x_tr, y_tr, x_test, y_test,
            k=k, weights=weights, metric=metric,
            pos_label=pos_label, neg_label=neg_label
        )

        # Store metrics
        accuracies[i] = accuracy
        sensitivities[i] = sensitivity
        specificities[i] = specificity
        precisions[i] = precision
        f1_scores[i] = f1
        roc_aucs[i] = roc_auc

    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1_scores),
        'f1_score_std': np.std(f1_scores),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }


In [153]:
distances = [('distance', 'minkowski'), ('distance', 'euclidean'), ('distance', 'manhattan'),
             ('uniform', 'minkowski'), ('uniform', 'euclidean'), ('uniform', 'manhattan')]
for i in range(1, 40, 2):
    for weights, metric in distances:
        print(f'--- KNN Classifier (weights={weights}, metric={metric}), k={i} ---')
        print(knn_c_full_test(blood_data, k=i, weights=weights, metric=metric))
print('--- KNN Classifier with PCA Components (k=5, weights=distance, metric=manhattan) ---')
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]
for i in range(1, n_features + 1):
    pca = fit_pca_reference(blood_data.drop(columns=[LABELS_KEY]), n_components=i)
    pca_comp, _, _ = pca_feature_reduction(blood_data, n_components=i, pca=pca)
    pca_data = pd.DataFrame(pca_comp)
    pca_data[LABELS_KEY] = blood_data[LABELS_KEY].to_numpy()
    print(f'--- PCA with {i} components ---')
    print(knn_c_full_test(pca_data, k=5, weights='distance', metric='manhattan'))

print('--- KNN Classifier with Kruskal-Wallis Features (k=5, weights=distance, metric=manhattan) ---')
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data_subset = blood_data[selected_features]
    print(f'--- Top {i} Kruskal features ---')
    print(knn_c_full_test(data_subset, k=5, weights='distance', metric='manhattan'))

--- KNN Classifier (weights=distance, metric=minkowski), k=1 ---
{'accuracy_mean': 0.6659047619047618, 'accuracy_std': 0.06439222217507325, 'sensitivity_mean': 0.6658759574725788, 'sensitivity_std': 0.10473603853000572, 'specificity_mean': 0.6724448476932997, 'specificity_std': 0.1079716085812557, 'precision_mean': 0.7138667821121111, 'precision_std': 0.09222169930394669, 'f1_score_mean': 0.6819686119722602, 'f1_score_std': 0.07128321381504225, 'roc_auc_mean': 0.6691604025829393, 'roc_auc_std': 0.06445985878629497}
--- KNN Classifier (weights=distance, metric=euclidean), k=1 ---
{'accuracy_mean': 0.6659047619047618, 'accuracy_std': 0.06439222217507325, 'sensitivity_mean': 0.6658759574725788, 'sensitivity_std': 0.10473603853000572, 'specificity_mean': 0.6724448476932997, 'specificity_std': 0.1079716085812557, 'precision_mean': 0.7138667821121111, 'precision_std': 0.09222169930394669, 'f1_score_mean': 0.6819686119722602, 'f1_score_std': 0.07128321381504225, 'roc_auc_mean': 0.669160402582

In [154]:
# Aplicar LDA ao dataset completo (usando train_data como referência)
lda_comp, _, _ = lda_feature_reduction(blood_data.drop(columns=[LABELS_KEY]), train_data)
lda_data = pd.DataFrame(lda_comp)
lda_data[LABELS_KEY] = blood_data[LABELS_KEY].to_numpy()

print('--- KNN Classifier with LDA Components (k=5, weights=distance, metric=manhattan) ---')
knn_results = knn_c_full_test(lda_data, k=5, weights='distance', metric='manhattan')
print(knn_results)

--- KNN Classifier with LDA Components (k=5, weights=distance, metric=manhattan) ---
{'accuracy_mean': 0.5518095238095239, 'accuracy_std': 0.06826412622824096, 'sensitivity_mean': 0.5968841737867583, 'sensitivity_std': 0.12186424534039061, 'specificity_mean': 0.5079129533177521, 'specificity_std': 0.1303775767346989, 'precision_mean': 0.5975911171457678, 'precision_std': 0.09529662980299176, 'f1_score_mean': 0.5877431175648763, 'f1_score_std': 0.07860615840218556, 'roc_auc_mean': 0.609910170601085, 'roc_auc_std': 0.08472036035601334}


## State Vector Machine (SVM)

In [161]:
from sklearn import svm


def run_svm(x_tr, y_tr, x_test, y_test, kernel='linear', C=1.0, gamma='scale', degree=3, probability=False):
    """
    Train and evaluate SVM classifier on single train/test split.
    
    Args:
        x_tr: Training features
        y_tr: Training labels
        x_test: Test features
        y_test: Test labels
        kernel: Kernel type ('linear', 'rbf', 'poly', 'sigmoid')
        C: Regularization parameter
        gamma: Kernel coefficient
        degree: Degree for polynomial kernel
        probability: If True, return probabilities; if False, return metrics
    
    Returns:
        If probability=True: predicted probabilities for positive class
        If probability=False: tuple of (accuracy, sensitivity, specificity, precision, f1)
    """
    svm_model = svm.SVC(kernel=kernel, C=C, gamma=gamma, degree=degree, probability=probability)
    svm_model.fit(x_tr, y_tr)
    
    if probability:
        y_proba = svm_model.predict_proba(x_test)[:, 1]
        return y_proba
    
    y_pred = svm_model.predict(x_test)
    return calculate_metrics(y_test, y_pred)


def svm_c_full_test(data, reduction_model=None, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150,
                    kernel='linear', C=1.0, gamma='auto', degree=2, probability=False):
    """
    Evaluate SVM classifier using Monte Carlo cross-validation.
    
    Trains and tests SVM multiple times with different random splits,
    optionally applying dimensionality reduction (PCA or LDA).
    
    Args:
        data: Complete dataset with features and labels
        reduction_model: Optional dict for dimensionality reduction
                        e.g., {'reduction':'PCA', 'n_components':5}
        labels_key: Name of the label column
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        n_times: Number of Monte Carlo iterations
        kernel: Kernel type ('linear', 'rbf', 'poly', 'sigmoid')
        C: Regularization parameter
        gamma: Kernel coefficient
        degree: Degree for polynomial kernel
        probability: Enable probability estimates
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    # Initialize metric arrays
    accuracies = np.zeros(n_times)
    sensitivities = np.zeros(n_times)
    specificities = np.zeros(n_times)
    precisions = np.zeros(n_times)
    f1_scores = np.zeros(n_times)
    roc_aucs = np.zeros(n_times)

    for i in range(n_times):
        # Generate train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(
            data,
            random_state=RAND_SEED + i,
            labels_key=labels_key,
            train_size=train_size,
            test_size=test_size,
        )

        # Apply dimensionality reduction if specified
        if reduction_model is not None:
            method = reduction_model.get('reduction', 'PCA')
            n_components = reduction_model.get('n_components', None)
            if method == 'PCA':
                pca = fit_pca_reference(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]), n_components=n_components)
                x_tr = pca.transform(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]))
                x_test = pca.transform(pd.DataFrame(x_test, columns=[c for c in data.columns if c != labels_key]))
            elif method == 'LDA':
                lda = LinearDiscriminantAnalysis(n_components=n_components)
                lda.fit(x_tr, y_tr)
                x_tr = lda.transform(x_tr)
                x_test = lda.transform(x_test)
            else:
                raise ValueError("Unknown reduction method. Use 'PCA' or 'LDA'.")

        # Train SVM and calculate metrics
        accuracies[i], sensitivities[i], specificities[i], precisions[i], f1_scores[i] = run_svm(
            x_tr, y_tr, x_test, y_test, kernel=kernel, C=C, gamma=gamma, degree=degree, probability=probability
        )

        # Get probabilities for ROC-AUC calculation
        probabilities = run_svm(
            x_tr, y_tr, x_test, y_test, kernel=kernel, C=C, gamma=gamma, degree=degree, probability=True
        )
        roc_aucs[i] = roc_auc_score(y_test, probabilities)

    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1_scores),
        'f1_score_std': np.std(f1_scores),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }


In [156]:
# Hyperparameter grids optimized for breast cancer classification (116 samples)
C_values = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0]

# RBF kernel: most sensitive to gamma, needs comprehensive search
gamma_rbf = ['scale', 'auto', 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]

# Polynomial kernel: typically works better with lower gamma values
gamma_poly = ['scale', 0.001, 0.005, 0.01, 0.05, 0.1]
degree_poly = [2, 3, 4]  # Higher degrees risk overfitting with small dataset

# Sigmoid kernel: benefits from wider gamma range
gamma_sigmoid = ['scale', 0.001, 0.005, 0.01, 0.05, 0.1, 0.5]

# Storage for all results
all_svm_results = []

print('=== SVM Linear Kernel (C sweep) ===')
for C in C_values:
    print(f'--- C={C} ---')
    result = svm_c_full_test(blood_data, kernel='linear', C=C)
    print(result)
    all_svm_results.append({
        'kernel': 'linear',
        'C': C,
        'gamma': None,
        'degree': None,
        **result
    })
    print()

print('=== SVM RBF Kernel (C x gamma) ===')
# Use subset for quick initial exploration
C_rbf = [0.1, 1.0, 10.0, 50.0, 100.0]
for C in C_rbf:
    for gamma in gamma_rbf:
        print(f'--- C={C}, gamma={gamma} ---')
        result = svm_c_full_test(blood_data, kernel='rbf', C=C, gamma=gamma)
        print(result)
        all_svm_results.append({
            'kernel': 'rbf',
            'C': C,
            'gamma': gamma,
            'degree': None,
            **result
        })
        print()

print('=== SVM Polynomial Kernel (C x degree x gamma) ===')
# Use reduced grid to avoid explosion of combinations
C_poly = [0.1, 1.0, 10.0, 50.0]
for C in C_poly:
    for degree in degree_poly:
        for gamma in gamma_poly:
            print(f'--- C={C}, degree={degree}, gamma={gamma} ---')
            result = svm_c_full_test(blood_data, kernel='poly', C=C, degree=degree, gamma=gamma)
            print(result)
            all_svm_results.append({
                'kernel': 'poly',
                'C': C,
                'gamma': gamma,
                'degree': degree,
                **result
            })
            print()

print('=== SVM Sigmoid Kernel (C x gamma) ===')
C_sigmoid = [0.1, 1.0, 10.0, 50.0]
for C in C_sigmoid:
    for gamma in gamma_sigmoid:
        print(f'--- C={C}, gamma={gamma} ---')
        result = svm_c_full_test(blood_data, kernel='sigmoid', C=C, gamma=gamma)
        print(result)
        all_svm_results.append({
            'kernel': 'sigmoid',
            'C': C,
            'gamma': gamma,
            'degree': None,
            **result
        })
        print()


=== SVM Linear Kernel (C sweep) ===
--- C=0.01 ---
{'accuracy_mean': 0.5318095238095237, 'accuracy_std': 0.07686274836465193, 'sensitivity_mean': 0.892579497263479, 'sensitivity_std': 0.2564947626988729, 'specificity_mean': 0.15464997964997962, 'specificity_std': 0.3254646414289369, 'precision_mean': 0.5842647194740097, 'precision_std': 0.1552464698507428, 'f1_score_mean': 0.6524593304234262, 'f1_score_std': 0.13547847749500427, 'roc_auc_mean': 0.7384751370463747, 'roc_auc_std': 0.08263120009011789}

--- C=0.05 ---
{'accuracy_mean': 0.665904761904762, 'accuracy_std': 0.0833937739999223, 'sensitivity_mean': 0.6800604742465424, 'sensitivity_std': 0.1371621628400858, 'specificity_mean': 0.6818598356121576, 'specificity_std': 0.19861881931035008, 'precision_mean': 0.7350725171331414, 'precision_std': 0.14056783730159947, 'f1_score_mean': 0.6863136669315842, 'f1_score_std': 0.07850747024981172, 'roc_auc_mean': 0.750023924162408, 'roc_auc_std': 0.07825062695041761}

--- C=0.1 ---
{'accuracy_

In [163]:
# Apply SVM on PCA components
print('--- SVM with PCA Components ---')
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]
for i in range(1, n_features + 1):
    print(f'--- PCA with {i} components ---')
    print(svm_c_full_test(
        blood_data,
        reduction_model={'reduction':'PCA', 'n_components':i},
        kernel='rbf', C=100.0, gamma=0.01
    ))

print('--- SVM with Kruskal-Wallis Features ---')
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data_subset = blood_data[selected_features]
    print(f'--- Top {i} Kruskal features ---')
    print(svm_c_full_test(
        data_subset,
        kernel='rbf', C=100.0, gamma=0.01
    ))

--- SVM with PCA Components ---
--- PCA with 1 components ---
{'accuracy_mean': 0.5328571428571427, 'accuracy_std': 0.06956130756537057, 'sensitivity_mean': 0.7726999127089986, 'sensitivity_std': 0.19539206823756133, 'specificity_mean': 0.30932307437144896, 'specificity_std': 0.24394097021405672, 'precision_mean': 0.5645631166668851, 'precision_std': 0.11600363294794915, 'f1_score_mean': 0.6258475399163366, 'f1_score_std': 0.06449487661726559}
--- PCA with 2 components ---
{'accuracy_mean': 0.5957142857142856, 'accuracy_std': 0.06011892975000233, 'sensitivity_mean': 0.665197104984425, 'sensitivity_std': 0.15057615321549575, 'specificity_mean': 0.5442269937664674, 'specificity_std': 0.19086239705525443, 'precision_mean': 0.6278601766092855, 'precision_std': 0.1068408059864411, 'f1_score_mean': 0.6275733775697047, 'f1_score_std': 0.059894016241160884}
--- PCA with 3 components ---
{'accuracy_mean': 0.6457142857142857, 'accuracy_std': 0.07740695533796613, 'sensitivity_mean': 0.73644268944

In [175]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

def run_decision_tree(x_tr, y_tr, x_test, y_test, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=RAND_SEED, return_proba=False):
    dt_model = DecisionTreeClassifier(
        criterion=criterion, 
        max_depth=max_depth, 
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state
    )
    dt_model.fit(x_tr, y_tr)
    y_pred = dt_model.predict(x_test)
    
    if return_proba:
        y_proba = dt_model.predict_proba(x_test)
        return calculate_metrics(y_test, y_pred), y_proba
    
    return calculate_metrics(y_test, y_pred)

def dt_c_full_test(data, reduction_method=None, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=150, criterion=None, max_depth=None, min_samples_split=None, min_samples_leaf=None):
    """
    Executa o Decision Tree em todo o conjunto de dados múltiplas vezes e retorna as métricas médias.
    Args:
        data (pd.DataFrame): Conjunto de dados completo.
        labels_key (str): Nome da coluna de rótulos.
        n_times (int): Número de vezes para repetir o teste.
        criterion (str): Critério de divisão ('gini' ou 'entropy').
        max_depth (int): Profundidade máxima da árvore.
        min_samples_split (int): Número mínimo de amostras para dividir um nó.
        min_samples_leaf (int): Número mínimo de amostras em um nó folha.
    """
    accuracies = np.zeros(n_times)
    sensitivities = np.zeros(n_times)
    specificities = np.zeros(n_times)
    precisions = np.zeros(n_times)
    f1_scores = np.zeros(n_times)
    roc_aucs = np.zeros(n_times)

    for i in range(n_times):
        x_tr, y_tr, x_test, y_test = generate_testing_data(data, random_state=RAND_SEED + i, labels_key=labels_key, train_size=train_size, test_size=test_size)

        if reduction_method is not None:
            method = reduction_method.get('reduction', 'PCA')
            n_components = reduction_method.get('n_components', None)
            if method == 'PCA':
                pca = fit_pca_reference(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]), n_components=n_components)
                x_tr = pca.transform(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]))
                x_test = pca.transform(pd.DataFrame(x_test, columns=[c for c in data.columns if c != labels_key]))
            elif method == 'LDA':
                lda = LinearDiscriminantAnalysis(n_components=n_components)
                lda.fit(x_tr, y_tr)
                x_tr = lda.transform(x_tr)
                x_test = lda.transform(x_test)
            else:
                raise ValueError("Unknown reduction method. Use 'PCA' or 'LDA'.")

        accuracies[i], sensitivities[i], specificities[i], precisions[i], f1_scores[i] = run_decision_tree(
            x_tr, y_tr, x_test, y_test, 
            criterion=criterion, 
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=RAND_SEED + i
        )
        # Calcular ROC-AUC com probabilidades
        metrics_proba, y_proba = run_decision_tree(
            x_tr, y_tr, x_test, y_test,
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=RAND_SEED + i,
            return_proba=True
        )
        roc_aucs[i] = roc_auc_score(y_test, y_proba[:, 1])

    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1_scores),
        'f1_score_std': np.std(f1_scores),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }

In [177]:
dt_configs = [
    {
        'criterion': 'gini',
        'max_depth': 3,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'criterion': 'entropy',
        'max_depth': 5,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'criterion': 'gini',
        'max_depth': 5,
        'min_samples_split': 5,
        'min_samples_leaf': 2
    },
    {
        'criterion': 'entropy',
        'max_depth': 7,
        'min_samples_split': 5,
        'min_samples_leaf': 2
    },
    {
        'criterion': 'gini',
        'max_depth': 7,
        'min_samples_split': 2,
        'min_samples_leaf': 1
    },
    {
        'criterion': 'entropy',
        'max_depth': 10,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'criterion': 'gini',
        'max_depth': None,
        'min_samples_split': 20,
        'min_samples_leaf': 10
    }
]

for config in dt_configs:
    print(config)
    result = dt_c_full_test(blood_data, **config)
    print(result)

{'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 5}
{'accuracy_mean': 0.683047619047619, 'accuracy_std': 0.09109144193419487, 'sensitivity_mean': 0.7053366442419747, 'sensitivity_std': 0.13551564647369174, 'specificity_mean': 0.6619577258253728, 'specificity_std': 0.17572447407945788, 'precision_mean': 0.726359210311498, 'precision_std': 0.12025529498049613, 'f1_score_mean': 0.7037065999891277, 'f1_score_std': 0.09730381583086918, 'roc_auc_mean': 0.7204888989877793, 'roc_auc_std': 0.09511145978786463}
{'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 5}
{'accuracy_mean': 0.6836190476190477, 'accuracy_std': 0.08314214803603136, 'sensitivity_mean': 0.7076404182925652, 'sensitivity_std': 0.12309716449310248, 'specificity_mean': 0.6631601722151258, 'specificity_std': 0.1424106469047988, 'precision_mean': 0.7225046772477484, 'precision_std': 0.10792553088032653, 'f1_score_mean': 0.7059868933681414, 'f1_score_std': 0.08672

In [178]:
# run on PCA components
print('--- Decision Tree with PCA Components ---')
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]
for i in range(1, n_features + 1):
    print(f'--- PCA with {i} components ---')
    print(dt_c_full_test(
        blood_data,
        reduction_method={'reduction':'PCA', 'n_components':i},
        criterion='gini',
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5
    ))

--- Decision Tree with PCA Components ---
--- PCA with 1 components ---
{'accuracy_mean': 0.5457142857142856, 'accuracy_std': 0.07104659772022195, 'sensitivity_mean': 0.6376028498838971, 'sensitivity_std': 0.19653391518292304, 'specificity_mean': 0.45056709544325635, 'specificity_std': 0.19916500805630652, 'precision_mean': 0.5901015295171402, 'precision_std': 0.09352628886776478, 'f1_score_mean': 0.5910505673968209, 'f1_score_std': 0.10417007750301588, 'roc_auc_mean': 0.5782276348489074, 'roc_auc_std': 0.08265867129148698}
--- PCA with 2 components ---
{'accuracy_mean': 0.5906666666666667, 'accuracy_std': 0.07873120839034022, 'sensitivity_mean': 0.6667194560593134, 'sensitivity_std': 0.1492557043518026, 'specificity_mean': 0.5045612475759536, 'specificity_std': 0.1558186735377871, 'precision_mean': 0.6237391410479144, 'precision_std': 0.08811730160757981, 'f1_score_mean': 0.6334465186867603, 'f1_score_std': 0.08895895377073396, 'roc_auc_mean': 0.6183615060666517, 'roc_auc_std': 0.0948

## Adaboost Classifier

In [185]:
def run_adaboost_classifier(x_tr, y_tr, x_test, y_test, n_estimators=10, max_depth=1, return_scores=False):
    """
    Train and evaluate AdaBoost classifier on single train/test split.
    
    AdaBoost (Adaptive Boosting) trains weak classifiers sequentially, with each
    new classifier focusing on samples misclassified by previous ones.
    
    Args:
        x_tr: Training features
        y_tr: Training labels (1/2 format)
        x_test: Test features
        y_test: Test labels (1/2 format)
        n_estimators: Number of weak classifiers (decision trees)
        max_depth: Maximum depth of each decision tree
        return_scores: If True, also return decision scores
    
    Returns:
        tuple: (metrics, decision_scores) if return_scores=True
               metrics only otherwise
        metrics = (accuracy, sensitivity, specificity, precision, f1)
    """
    # Convert labels from 1/2 to -1/1 format for AdaBoost
    y_tr_adaboost = np.where(y_tr == 2, 1, -1).astype(float)
    y_test_adaboost = np.where(y_test == 2, 1, -1).astype(float)
    
    def decision_tree_pred(x, y, weights):
        """Train a single decision tree with given weights."""
        clf = DecisionTreeClassifier(max_depth=max_depth)
        clf = clf.fit(x, y, weights)
        return clf.predict(x), clf
    
    # Initialize weights uniformly
    N = x_tr.shape[0]
    weights = np.full(N, 1/N)
    alphas = np.array([])
    classifiers = np.array([])

    # Train weak classifiers sequentially
    for _ in range(n_estimators):
        # Train weak classifier
        H_m, clf = decision_tree_pred(x_tr, y_tr_adaboost, weights)
        
        # Calculate weighted error
        err = sum(weights * np.heaviside(- y_tr_adaboost * H_m, 0))
        
        # Avoid division by zero
        err = np.clip(err, 1e-10, 1 - 1e-10)
        
        # Calculate classifier weight (alpha)
        alpha_m = 0.5 * np.log((1 - err) / err)
        alphas = np.append(alphas, alpha_m)
        classifiers = np.append(classifiers, clf)
        
        # Update sample weights (increase weight for misclassified samples)
        for i in range(N):
            weights[i] = weights[i] * np.exp(- alpha_m * y_tr_adaboost[i] * H_m[i])
        weights = weights / sum(weights)
    
    # Make predictions using weighted vote
    decision_scores = np.sum([alpha * clf.predict(x_test) for alpha, clf in zip(alphas, classifiers)], axis=0)
    predictions_adaboost = np.sign(decision_scores)
    
    # Convert predictions back to 1/2 label format
    predictions = np.where(predictions_adaboost == 1, 2, 1)
    
    return calculate_metrics(y_test, predictions), decision_scores


def adaboost_c_full_test(data, reduction_method=None, labels_key=LABELS_KEY, train_size=TRAIN_SIZE, test_size=TEST_SIZE, n_times=20, n_estimators=10, max_depth=1):
    """
    Evaluate AdaBoost classifier using Monte Carlo cross-validation.
    
    Trains and tests AdaBoost multiple times with different random splits,
    optionally applying dimensionality reduction.
    
    Args:
        data: Complete dataset with features and labels
        reduction_method: Optional dict for dimensionality reduction
                         e.g., {'reduction':'PCA', 'n_components':5}
        labels_key: Name of the label column
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        n_times: Number of Monte Carlo iterations
        n_estimators: Number of weak classifiers
        max_depth: Maximum depth of each decision tree
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    # Initialize metric arrays
    accuracies = np.zeros(n_times)
    sensitivities = np.zeros(n_times)
    specificities = np.zeros(n_times)
    precisions = np.zeros(n_times)
    f1_scores = np.zeros(n_times)
    roc_aucs = np.zeros(n_times)

    for i in range(n_times):
        # Generate train/test split
        x_tr, y_tr, x_test, y_test = generate_testing_data(data, random_state=RAND_SEED + i, labels_key=labels_key, train_size=train_size, test_size=test_size)
        
        # Apply dimensionality reduction if specified
        if reduction_method is not None:
            method = reduction_method.get('reduction', 'PCA')
            n_components = reduction_method.get('n_components', None)
            if method == 'PCA':
                pca = fit_pca_reference(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]), n_components=n_components)
                x_tr = pca.transform(pd.DataFrame(x_tr, columns=[c for c in data.columns if c != labels_key]))
                x_test = pca.transform(pd.DataFrame(x_test, columns=[c for c in data.columns if c != labels_key]))
            elif method == 'LDA':
                lda = LinearDiscriminantAnalysis(n_components=n_components)
                lda.fit(x_tr, y_tr)
                x_tr = lda.transform(x_tr)
                x_test = lda.transform(x_test)
            else:
                raise ValueError("Unknown reduction method. Use 'PCA' or 'LDA'.")
        
        # Train and evaluate AdaBoost
        metrics, decision_scores = run_adaboost_classifier(
            x_tr, y_tr, x_test, y_test, 
            n_estimators=n_estimators,
            max_depth=max_depth
        )
        accuracies[i], sensitivities[i], specificities[i], precisions[i], f1_scores[i] = metrics
        
        # Convert labels from 1/2 to 0/1 for ROC-AUC calculation
        y_test_binary = np.where(y_test == 2, 1, 0)
        roc_aucs[i] = roc_auc_score(y_test_binary, decision_scores)

    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1_scores),
        'f1_score_std': np.std(f1_scores),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }


In [183]:
# run on whole data with different n_estimators and max_depth
adaboost_estimaors = range(10, 70, 10)
adaboost_depths = [1, 2, 3, 5, 6, 7, 10]

for n_estimators in adaboost_estimaors:
    for max_depth in adaboost_depths:
        print(f'--- AdaBoost with n_estimators={n_estimators}, max_depth={max_depth} ---')
        result = adaboost_c_full_test(
            blood_data,
            n_estimators=n_estimators,
            max_depth=max_depth
        )
        print(result)

--- AdaBoost with n_estimators=10, max_depth=1 ---
{'accuracy_mean': 0.6985714285714286, 'accuracy_std': 0.07254836528911215, 'sensitivity_mean': 0.7262489775745261, 'sensitivity_std': 0.1378482497747177, 'specificity_mean': 0.6826587726858625, 'specificity_std': 0.12127451669972109, 'precision_mean': 0.7194757113296956, 'precision_std': 0.09500395815544872, 'f1_score_mean': 0.7116553472198633, 'f1_score_std': 0.08090084261079736, 'roc_auc_mean': 0.7808835785861233, 'roc_auc_std': 0.0623754739920491}
--- AdaBoost with n_estimators=10, max_depth=2 ---
{'accuracy_mean': 0.7185714285714286, 'accuracy_std': 0.060118929750002346, 'sensitivity_mean': 0.7646058246299698, 'sensitivity_std': 0.12549179195134044, 'specificity_mean': 0.6827458517145049, 'specificity_std': 0.10070864704987716, 'precision_mean': 0.7288972524798006, 'precision_std': 0.07779627788532073, 'f1_score_mean': 0.7364109682537712, 'f1_score_std': 0.06420176145905526, 'roc_auc_mean': 0.803448699616613, 'roc_auc_std': 0.06115

In [186]:
# Run Adaboost on PCA components
print('--- AdaBoost with PCA Components ---')
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]
for i in range(1, n_features + 1):
    print(f'--- PCA with {i} components ---')
    print(adaboost_c_full_test(
        blood_data,
        reduction_method={'reduction':'PCA', 'n_components':i},
        n_estimators=20,
        max_depth=3
    ))

--- AdaBoost with PCA Components ---
--- PCA with 1 components ---
{'accuracy_mean': 0.5442857142857143, 'accuracy_std': 0.071983558213395, 'sensitivity_mean': 0.6395873199671659, 'sensitivity_std': 0.12491433133042903, 'specificity_mean': 0.45177877055005544, 'specificity_std': 0.17096453474277998, 'precision_mean': 0.5718310550367042, 'precision_std': 0.10362133711389765, 'f1_score_mean': 0.5918887377165526, 'f1_score_std': 0.0669513635072947, 'roc_auc_mean': 0.5998742718229803, 'roc_auc_std': 0.08316958307015962}
--- PCA with 2 components ---
{'accuracy_mean': 0.5914285714285714, 'accuracy_std': 0.10265853841840963, 'sensitivity_mean': 0.6826581927237297, 'sensitivity_std': 0.13885243318972995, 'specificity_mean': 0.4906153038138332, 'specificity_std': 0.11293617911288101, 'precision_mean': 0.5952135522435865, 'precision_std': 0.10106117372589485, 'f1_score_mean': 0.6312870648340783, 'f1_score_std': 0.10991707466699858, 'roc_auc_mean': 0.6490778982960527, 'roc_auc_std': 0.1079316799

In [187]:
# Run Adaboost on Kruskall-Wallis selected features
print('--- AdaBoost with Kruskal-Wallis Features ---')
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data_subset = blood_data[selected_features]
    print(f'--- Top {i} Kruskal features ---')
    print(adaboost_c_full_test(
        data_subset,
        n_estimators=20,
        max_depth=3
    ))

--- AdaBoost with Kruskal-Wallis Features ---
--- Top 1 Kruskal features ---
{'accuracy_mean': 0.5957142857142856, 'accuracy_std': 0.05662263195030406, 'sensitivity_mean': 0.6409346176755149, 'sensitivity_std': 0.11891395043703705, 'specificity_mean': 0.5639846836170366, 'specificity_std': 0.14326783555013836, 'precision_mean': 0.6243453054646955, 'precision_std': 0.09757433308466282, 'f1_score_mean': 0.6201924078437963, 'f1_score_std': 0.06273454911801224, 'roc_auc_mean': 0.6510779386305257, 'roc_auc_std': 0.0652764737127088}
--- Top 2 Kruskal features ---
{'accuracy_mean': 0.5928571428571427, 'accuracy_std': 0.08225148097419875, 'sensitivity_mean': 0.6618150499572966, 'sensitivity_std': 0.14176235361198272, 'specificity_mean': 0.5285872675655956, 'specificity_std': 0.1293868109163507, 'precision_mean': 0.612157539449453, 'precision_std': 0.07515123494468512, 'f1_score_mean': 0.6260603048417694, 'f1_score_std': 0.07525286304397524, 'roc_auc_mean': 0.6557708825701153, 'roc_auc_std': 0.

## Random Forest

In [201]:
from sklearn.ensemble import RandomForestClassifier

def run_random_forest_classifier(x_train, y_train, x_test, y_test, 
                                 n_estimators=100, max_depth=None, 
                                 min_samples_split=2, min_samples_leaf=1, 
                                 random_state=RAND_SEED, return_proba=False):
    """
    Train and evaluate Random Forest classifier on single train/test split.
    
    Random Forest is an ensemble learning method that constructs multiple decision trees
    during training and outputs the mode (majority vote) of individual tree predictions.
    It reduces overfitting compared to single decision trees through bagging and random
    feature selection at each split.
    
    Algorithm Overview:
    1. Bootstrap sampling: Create n_estimators different training subsets (with replacement)
    2. For each bootstrap sample:
       a. Build decision tree
       b. At each split, consider only random subset of features
    3. Prediction: Majority vote (classification) across all trees
    
    Key Advantages:
    - Reduces variance through ensemble averaging
    - Handles non-linear relationships and feature interactions
    - Built-in feature importance estimates
    - Robust to outliers and noise
    - Less prone to overfitting than single decision trees
    
    Args:
        x_train: Training features, shape (n_train, n_features)
        y_train: Training labels, shape (n_train,)
        x_test: Test features, shape (n_test, n_features)
        y_test: Test labels, shape (n_test,)
        n_estimators: Number of trees in forest. Default: 100
                     More trees: Better performance but slower
        max_depth: Maximum depth of each tree. Default: None (unlimited)
                  If None: Trees grow until leaves are pure
                  If int: Limits tree depth (regularization)
        min_samples_split: Minimum samples required to split internal node. Default: 2
                          Higher values: More regularization
        min_samples_leaf: Minimum samples required at leaf node. Default: 1
                         Higher values: Smoother decision boundaries
        random_state: Random seed for reproducibility
        return_proba: If True, also return predicted probabilities
    
    Returns:
        tuple: ((accuracy, sensitivity, specificity, precision, f1_score), y_proba)
            If return_proba=False: Only returns metrics tuple
            If return_proba=True: Returns (metrics_tuple, probabilities_array)
    """
    # Create and train the Random Forest classifier
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state
    )
    rf.fit(x_train, y_train)
    
    # Make predictions
    y_pred = rf.predict(x_test)
    
    # Calculate classification metrics
    accuracy, sensitivity, specificity, precision, f1 = calculate_metrics(y_test, y_pred)
    
    # Get predicted probabilities for positive class (used for ROC-AUC)
    y_proba = rf.predict_proba(x_test)[:, 1]
    
    return (accuracy, sensitivity, specificity, precision, f1), y_proba


def rf_c_full_test(data, reduction_method=None, labels_key=LABELS_KEY,
                   n_estimators=100, max_depth=None, 
                   min_samples_split=2, min_samples_leaf=1,
                   n_times=150, train_size=TRAIN_SIZE, test_size=TEST_SIZE,
                   random_state=RAND_SEED):
    """
    Evaluate Random Forest classifier using Monte Carlo cross-validation.
    
    Trains and tests Random Forest multiple times with different random splits,
    optionally applying dimensionality reduction.
    
    Args:
        data: Complete dataset with features and labels
        reduction_method: Optional dict for dimensionality reduction
                         e.g., {'reduction':'PCA', 'n_components':5}
        labels_key: Name of the label column
        n_estimators: Number of trees in forest
        max_depth: Maximum depth of each tree
        min_samples_split: Minimum samples to split internal node
        min_samples_leaf: Minimum samples at leaf node
        n_times: Number of Monte Carlo iterations
        train_size: Proportion of data for training
        test_size: Proportion of data for testing
        random_state: Random seed for reproducibility
    
    Returns:
        Dictionary with mean and std of all metrics across iterations
    """
    # Initialize metric lists
    accuracies = []
    sensitivities = []
    specificities = []
    precisions = []
    f1_scores = []
    roc_aucs = []
    
    for i in range(n_times):
        # Generate train/test split
        x_train, y_train, x_test, y_test = generate_testing_data(
            data, labels_key=labels_key, random_state=random_state + i, 
            train_size=train_size, test_size=test_size
        )
        
        # Apply dimensionality reduction if specified
        if reduction_method is not None:
            if reduction_method['reduction'] == 'PCA':
                pca = fit_pca_reference(pd.DataFrame(x_train), reduction_method['n_components'])
                x_train = pca.transform(x_train)
                x_test = pca.transform(x_test)
            elif reduction_method['reduction'] == 'LDA':
                lda = LinearDiscriminantAnalysis(n_components=reduction_method['n_components'])
                x_train = lda.fit_transform(x_train, y_train)
                x_test = lda.transform(x_test)
        
        # Run classifier to get metrics and probabilities
        metrics, y_proba = run_random_forest_classifier(
            x_train, y_train, x_test, y_test,
            n_estimators=n_estimators, max_depth=max_depth,
            min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
            random_state=random_state + i, return_proba=True
        )
        
        accuracy, sensitivity, specificity, precision, f1 = metrics
        
        # Calculate ROC AUC
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Store metrics
        accuracies.append(accuracy)
        sensitivities.append(sensitivity)
        specificities.append(specificity)
        precisions.append(precision)
        f1_scores.append(f1)
        roc_aucs.append(roc_auc)
    
    # Return mean and std for each metric
    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'sensitivity_mean': np.mean(sensitivities),
        'sensitivity_std': np.std(sensitivities),
        'specificity_mean': np.mean(specificities),
        'specificity_std': np.std(specificities),
        'precision_mean': np.mean(precisions),
        'precision_std': np.std(precisions),
        'f1_score_mean': np.mean(f1_scores),
        'f1_score_std': np.std(f1_scores),
        'roc_auc_mean': np.mean(roc_aucs),
        'roc_auc_std': np.std(roc_aucs)
    }


In [195]:
# run on whole dataset using different hyperparameters
rf_n_estimators = [20, 30, 50, 70, 100]
rf_max_depths = [None, 2, 3, 4, 5, 8, 10]

for n_estimators in rf_n_estimators:
    for max_depth in rf_max_depths:
        print(f'--- Random Forest with n_estimators={n_estimators}, max_depth={max_depth} ---')
        result = rf_c_full_test(
            blood_data,
            n_estimators=n_estimators,
            max_depth=max_depth
        )
        print(result)

--- Random Forest with n_estimators=20, max_depth=None ---
{'accuracy_mean': 0.7066666666666667, 'accuracy_std': 0.06669387200004419, 'sensitivity_mean': 0.7160795635931052, 'sensitivity_std': 0.10233984742537824, 'specificity_mean': 0.7055101734064583, 'specificity_std': 0.12421598481621253, 'precision_mean': 0.7518938515710221, 'precision_std': 0.10143596526431735, 'f1_score_mean': 0.7251732995645634, 'f1_score_std': 0.06732778239900866, 'roc_auc_mean': 0.7915160800237195, 'roc_auc_std': 0.06753828012333986}
--- Random Forest with n_estimators=20, max_depth=2 ---
{'accuracy_mean': 0.703047619047619, 'accuracy_std': 0.06856322702275065, 'sensitivity_mean': 0.7770772197973973, 'sensitivity_std': 0.11092214965051692, 'specificity_mean': 0.6285380039350628, 'specificity_std': 0.13638223916019324, 'precision_mean': 0.7204825246928971, 'precision_std': 0.09668438207912046, 'f1_score_mean': 0.738673333516486, 'f1_score_std': 0.0646244305778672, 'roc_auc_mean': 0.788453351098762, 'roc_auc_st

In [202]:
# run on PCA components
print('--- Random Forest with PCA Components ---')
n_features = blood_data.drop(columns=[LABELS_KEY]).shape[1]
for i in range(1, n_features + 1):
    print(f'--- PCA with {i} components ---')
    print(rf_c_full_test(
        blood_data,
        reduction_method={'reduction':'PCA', 'n_components':i},
        n_estimators=30,
        max_depth=4
    ))

--- Random Forest with PCA Components ---
--- PCA with 1 components ---
{'accuracy_mean': 0.5283809523809524, 'accuracy_std': 0.0691835115952059, 'sensitivity_mean': 0.6453048026471367, 'sensitivity_std': 0.16122895647783897, 'specificity_mean': 0.40820117713383963, 'specificity_std': 0.15085778969634026, 'precision_mean': 0.569866409812557, 'precision_std': 0.08502518015233855, 'f1_score_mean': 0.5903602632774481, 'f1_score_std': 0.08758548433444749, 'roc_auc_mean': 0.5741540497628377, 'roc_auc_std': 0.08513121636697381}
--- PCA with 2 components ---
{'accuracy_mean': 0.5912380952380952, 'accuracy_std': 0.07413513431962993, 'sensitivity_mean': 0.7079496420885167, 'sensitivity_std': 0.13016319042206215, 'specificity_mean': 0.4661556381355143, 'specificity_std': 0.1438047991399595, 'precision_mean': 0.6201886642131379, 'precision_std': 0.08965399820614628, 'f1_score_mean': 0.6500057294526217, 'f1_score_std': 0.07273558107272354, 'roc_auc_mean': 0.651111667917121, 'roc_auc_std': 0.079555

In [203]:
# Run on Kruskal-Wallis selected features
print('--- Random Forest with Kruskal-Wallis Features ---')
for i in range(1, len(kruskal_ranking) + 1):
    selected_features = [feat_name for feat_name, _ in kruskal_ranking[:i]]
    selected_features.append(LABELS_KEY)
    data_subset = blood_data[selected_features]
    print(f'--- Top {i} Kruskal features ---')
    print(rf_c_full_test(
        data_subset,
        n_estimators=30,
        max_depth=4
    ))

--- Random Forest with Kruskal-Wallis Features ---
--- Top 1 Kruskal features ---
{'accuracy_mean': 0.6102857142857143, 'accuracy_std': 0.07047207195173184, 'sensitivity_mean': 0.6532746121857981, 'sensitivity_std': 0.1260565851041029, 'specificity_mean': 0.5685078651998157, 'specificity_std': 0.14956845161199397, 'precision_mean': 0.6526178392234989, 'precision_std': 0.10261148840517167, 'f1_score_mean': 0.6419645139980688, 'f1_score_std': 0.08152129866024907, 'roc_auc_mean': 0.6633145740557632, 'roc_auc_std': 0.06649399545554537}
--- Top 2 Kruskal features ---
{'accuracy_mean': 0.6388571428571428, 'accuracy_std': 0.06867294073443811, 'sensitivity_mean': 0.6973438075758912, 'sensitivity_std': 0.11638510907133076, 'specificity_mean': 0.5770940662364812, 'specificity_std': 0.12672791190960495, 'precision_mean': 0.6691142527867928, 'precision_std': 0.09557995368742651, 'f1_score_mean': 0.6747450444795855, 'f1_score_std': 0.07431365980404535, 'roc_auc_mean': 0.6888741620150859, 'roc_auc_s

In [204]:
# Test SVM with Glucose, HOMA, Insulin, BMI and Age
selected_features = ['Glucose', 'HOMA', 'Insulin', 'BMI', 'Age', LABELS_KEY]
data_subset = blood_data[selected_features]
print('--- SVM with selected features (Glucose, HOMA, Insulin, BMI, Age) ---')
print(svm_c_full_test(
    data_subset,
    kernel='rbf', C=100.0, gamma=0.01
))


--- SVM with selected features (Glucose, HOMA, Insulin, BMI, Age) ---
{'accuracy_mean': 0.7455238095238095, 'accuracy_std': 0.06812127374402169, 'sensitivity_mean': 0.7333617833717512, 'sensitivity_std': 0.10682744039957233, 'specificity_mean': 0.7719352664592602, 'specificity_std': 0.12799648033184144, 'precision_mean': 0.7992429429327396, 'precision_std': 0.11026680877040863, 'f1_score_mean': 0.7558675637113013, 'f1_score_std': 0.07436501119625398, 'roc_auc_mean': 0.8204286965577153, 'roc_auc_std': 0.06533106785127363}
